# Prostate Cancer Grading — Graph Transformer + ILRA-MIL

**Pipeline:** DINOv2 patch features -> multi-scale slide graphs -> Graph
Transformer with ILRA-MIL pooling -> ISUP grade. Trained on **SICAPv2**,
externally validated on **PANDA**.

This notebook runs all 17 stages in sequence. Every stage writes its outputs to
`/kaggle/working/` and prints a completion message. The behaviour (epoch counts,
slide subsets, bootstrap iterations) is controlled entirely by the `SMOKE_TEST`
flag in the **CONFIG** cell below.

> **Requirements before running:** enable *Internet* (DINOv2 hub download + pip),
> enable *GPU P100*, and attach both datasets at the exact paths used below.

## CONFIG — toggles the entire run

In [1]:
# ==================== CONFIG ====================
# This is the ONLY cell that differs between the smoke-test and full notebooks.
SMOKE_TEST = False   # <-- FULL run
print('SMOKE_TEST =', SMOKE_TEST)

SMOKE_TEST = False


## Stage 1 — Environment setup, seeds, paths, hyperparameters

In [2]:
# ---- Stage 1a: installs that DO NOT touch the Kaggle torch build ----------
# CRITICAL (P100 fix): do NOT pip-install torch, torch-scatter, torch-sparse or
# torch-cluster. Those wheels routinely replace Kaggle's torch with a build that
# has no CUDA kernels for the P100 (sm_60), causing:
#   "CUDA error: no kernel image is available for execution on the device".
# PyG's TransformerConv / global_mean_pool use torch-native ops, and the KNN
# graph has a numpy fallback, so the companion packages are NOT needed.
import os, sys, subprocess, importlib
# expandable segments reduces fragmentation OOMs (set before torch initialises CUDA)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

ORIG_TORCH = torch.__version__
print("Kaggle torch:", ORIG_TORCH, "| CUDA:", torch.version.cuda)

def _pip(*args):
    """Run pip quietly; never abort the notebook on a single failure."""
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)
    except Exception as e:
        print("pip warning:", e)

def _have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

# torch-geometric core is pure-python; --no-deps prevents any torch upgrade.
if not _have("torch_geometric"):
    _pip("--no-deps", "torch-geometric")
    for dep in ["fsspec", "aiohttp", "psutil"]:
        if not _have(dep):
            _pip(dep)

# staintools/spams frequently fail to build on Kaggle; we attempt it but also
# ship a pure-numpy Macenko fallback in Stage 3, so this is best-effort only.
if not _have("staintools"):
    _pip("spams-bin")        # prebuilt spams wheel, avoids source build
    _pip("staintools")
if not _have("openslide"):
    _pip("openslide-python")
# encoder libraries: timm (UNI / Prov-GigaPath), transformers (Phikon), hub auth
for _m in ("timm", "transformers", "huggingface_hub"):
    if not _have(_m):
        _pip(_m)
# explainability: Grad-CAM (pytorch_grad_cam) + LIME
for _m, _pkg in (("pytorch_grad_cam", "grad-cam"), ("lime", "lime")):
    if not _have(_m):
        _pip(_pkg)

# =====================================================================
# NOTEBOOK VERSION BANNER  ->  confirm you are running the UPDATED file
# =====================================================================
NOTEBOOK_VERSION = "v28  (encoder fine-tuning for binary/GP classifiers; PANDA sampling stays RANDOM)"
print("=" * 60)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("=" * 60)

# ---- GPU sanity check: catch the 'no kernel image' error HERE, not in Stage 6 ----
print("torch after installs:", torch.__version__,
      "(changed!)" if torch.__version__ != ORIG_TORCH else "(unchanged)")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    sm = f"sm_{cap[0]}{cap[1]}"
    try:
        archs = torch.cuda.get_arch_list()
    except Exception:
        archs = []
    print("GPU:", torch.cuda.get_device_name(0), "| compute capability:", sm)
    print("torch was built for archs:", archs)
    if archs and sm not in archs:
        print("\n" + "!" * 64)
        print(f"!! ARCH MISMATCH: this PyTorch has NO kernels for {sm} "
              f"({torch.cuda.get_device_name(0)}).")
        print("!! This is the cause of 'CUDA error: no kernel image is available'.")
        print("!! FIX -> right panel: Settings -> Accelerator -> 'GPU T4 x2'")
        print("!!        (Turing sm_75 IS supported), then Factory reset & re-run.")
        print("!" * 64 + "\n")
    # functional check using conv2d (the exact op that fails in DINOv2)
    try:
        _x = torch.randn(1, 3, 28, 28, device="cuda")
        _c = torch.nn.Conv2d(3, 4, 3).to("cuda")
        _ = _c(_x); torch.cuda.synchronize()
        del _x, _c; torch.cuda.empty_cache()
        _free, _tot = torch.cuda.mem_get_info()
        print(f"GPU conv check: OK  ->  free VRAM at start: {_free/1e9:.1f} / {_tot/1e9:.1f} GB")
    except Exception as e:
        print("GPU conv check FAILED:", e)
        print(">>> Switch Accelerator to 'GPU T4 x2', Factory reset, and re-run.")
else:
    print("No CUDA device visible -> enable a GPU accelerator (T4 x2 recommended).")
print("Stage 1a complete: installs attempted (torch left untouched).")

Kaggle torch: 2.10.0+cu128 | CUDA: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 81.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

NOTEBOOK VERSION: v28  (encoder fine-tuning for binary/GP classifiers; PANDA sampling stays RANDOM)
torch after installs: 2.10.0+cu128 (unchanged)
GPU: Tesla T4 | compute capability: sm_75
torch was built for archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU conv check: OK  ->  free VRAM at start: 15.5 / 15.6 GB
Stage 1a complete: installs attempted (torch left untouched).


In [3]:
# ---- Stage 1b: imports, seeds, paths, hyperparameters ---------------------
import os, random, math, json, gc, warnings, traceback, re
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

import torchvision
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
try:
    import torch_geometric
    print("torch-geometric:", torch_geometric.__version__)
except Exception as e:
    print("torch-geometric import failed:", e)

# ---- reproducibility ----
SEED = 42
def set_seed(seed=SEED):
    """Seed python, numpy and torch (incl. CUDA) for deterministic runs."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---- exact Kaggle input paths ----
# Structured (Mendeley) SICAPv2: flat images/ + masks/ with WSI-prefixed filenames,
# partition/ split files, and wsi_labels.xlsx (slide-level Gleason ground truth).
SICAP_ROOT   = Path("/kaggle/input/datasets/aminurjoyy/sicapv2/SICAPv2/SICAPv2")
SICAP_TR_IMG = SICAP_ROOT / "images"
SICAP_TR_MSK = SICAP_ROOT / "masks"
SICAP_VA_IMG = SICAP_ROOT / "images"      # unused: val is carved from images/ by slide
SICAP_VA_MSK = SICAP_ROOT / "masks"
SICAP_WSI_LABELS = SICAP_ROOT / "wsi_labels.xlsx"
SICAP_PARTITION  = SICAP_ROOT / "partition" / "Test"   # official Train.xlsx + Test.xlsx
USE_PARTITION_LABELS = True   # clean ground-truth patch labels (NC/G3/G4/G5) vs noisy masks

# ---- Gleason2019 (Route A: extra training cohort for cross-dataset generalisation) ----
GLEASON_ROOT = Path("/kaggle/input/datasets/danielerussica/gleason2019-grand-challenge")
GLEASON_IMG  = GLEASON_ROOT / "Train Imgs"
GLEASON_MASK = GLEASON_ROOT / "STAPLED_maps"   # multi-rater STAPLE consensus masks
USE_GLEASON  = True                            # add Gleason2019 cores to TRAINING (not val)
# Gleason2019 STAPLE encoding (confirmed from masks): 0=benign, 3/4/5=Gleason grades.
# NOTE this differs from SICAPv2 (0=background, 1=benign). Stage 2 prints unique values.
GLEASON_MASK_MAP = {0: "Benign", 3: "GP3", 4: "GP4", 5: "GP5"}
GLEASON_TILE = 512
PANDA_ROOT   = Path("/kaggle/input/competitions/prostate-cancer-grade-assessment")
PANDA_IMG    = PANDA_ROOT / "train_images"
PANDA_MSK    = PANDA_ROOT / "train_label_masks"
PANDA_CSV    = PANDA_ROOT / "train.csv"

# ---- working output root ----
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

# ---- hyperparameters ----
PATCH_SIZE       = 512
INPUT_SIZE       = 518
TISSUE_THRESHOLD = 0.05
KNN_K            = 8
EMBED_DIM        = 768
HIDDEN_DIM       = 512
NUM_HEADS        = 8
NUM_LAYERS       = 3
ILRA_LATENT      = 128
DROPOUT          = 0.1
BATCH_SIZE_PATCH = 16     # 16 fits a 16GB T4 at 518px; raise to 32 on bigger GPUs
FT_LAST_BLOCKS   = 2      # fine-tune only the last K ViT blocks (memory + small-data friendly)
BATCH_SIZE_GRAPH = 4
LR_HEAD          = 2e-3
LR_FINETUNE      = 1e-4
WEIGHT_DECAY     = 2e-3
EPOCHS_FROZEN    = 10
EPOCHS_UNFREEZE  = 10
EPOCHS_GRAPH     = 30
PATIENCE         = 4      # patch classifiers: train longer (was 2 -> under-trained)
PATIENCE_GRAPH   = 5
NUM_WORKERS      = 0      # single-process loading: avoids Kaggle DataLoader worker-shutdown crashes
N_FOLDS          = 5
N_BOOTSTRAP      = 4000

# ---- SMOKE_TEST overrides (SMOKE_TEST is defined in the CONFIG cell above) ----
# Caps that limit how much data the smoke run touches.
MAX_TRAIN_PATCHES = None
MAX_VAL_PATCHES   = None
MAX_PANDA_SLIDES  = 200
MAX_GRAPH_SLIDES  = None
MAX_TILES_PER_SLIDE = 384      # tiles per PANDA WSI (~47s/slide -> fits Kaggle 12h)
GRAPH_OBJECTIVE = "ordinal"    # "ordinal" (regression -> optimizes QWK) or "ce" (cross-entropy)
GLEASON_CAP_CORES = None       # cap Gleason2019 cores ingested (None = all)
GLEASON_CAP_TILES = 60         # max tiles per Gleason core (lower -> faster stain norm)
if SMOKE_TEST:
    EPOCHS_FROZEN, EPOCHS_UNFREEZE, EPOCHS_GRAPH = 1, 1, 2
    PATIENCE, PATIENCE_GRAPH = 1, 1
    N_FOLDS = 2
    N_BOOTSTRAP = 200
    MAX_TRAIN_PATCHES = 400
    MAX_VAL_PATCHES   = 200
    MAX_PANDA_SLIDES  = 8
    MAX_GRAPH_SLIDES  = 40
    MAX_TILES_PER_SLIDE = 48
    GLEASON_CAP_CORES = 6
    GLEASON_CAP_TILES = 20
    print(">>> SMOKE_TEST mode: minimal epochs and data subsets <<<")
else:
    # Larger RANDOM PANDA sample -> tighter external CI (caveat #1). ~600 slides
    # at ~45s each is ~7.5h; lower this if you hit Kaggle's runtime limit.
    MAX_PANDA_SLIDES = 300
    print(">>> FULL run mode <<<  (PANDA external test: "
          f"{MAX_PANDA_SLIDES} RANDOM slides, {MAX_TILES_PER_SLIDE} tiles/slide)")

CLASS_NAMES = ["Benign", "GP3", "GP4", "GP5"]
LABEL2INT = {n: i for i, n in enumerate(CLASS_NAMES)}

# ---- shared helpers used across stages ----
def get_slide_id(filename):
    """Slide id = filename token before the first underscore (SICAP convention)."""
    return Path(filename).stem.split("_")[0]

def get_coords(filename):
    """Patch grid (row,col). Prefer xini/yini tokens, else last two integers."""
    stem = Path(filename).stem
    m = re.search(r"xini[_]?(\d+).*?yini[_]?(\d+)", stem)
    if m:
        x, y = int(m.group(1)), int(m.group(2))
        return (y // PATCH_SIZE, x // PATCH_SIZE)
    nums = re.findall(r"\d+", stem)
    if len(nums) >= 2:
        return (int(nums[-2]), int(nums[-1]))
    return (0, 0)

def free_gpu(*names):
    """Drop named globals + clear CUDA cache to recover stale VRAM from a prior
    failed run. Prints free/total VRAM so you can see if a restart is needed."""
    for n in names:
        if n in globals():
            try:
                del globals()[n]
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"GPU free: {free/1e9:.1f} / {total/1e9:.1f} GB")
        return free / 1e9
    return None

# ======================================================================
# PATCH ENCODER SELECTION  (DINOv2 baseline vs pathology foundation models)
# ======================================================================
# Pathology FMs (UNI/GigaPath/Virchow) dramatically beat natural-image DINOv2 on
# H&E and are the key lever to surpass the base paper. Set ENCODER below.
#   dinov2   : facebook DINOv2 ViT-B/14 reg  (baseline, open, 768-d, 518px)
#   phikon   : Owkin Phikon  ViT-B           (OPEN, no token, 768-d, 224px)
#   phikon2  : Owkin Phikon-v2 ViT-L         (OPEN, no token, 1024-d, 224px)
#   uni      : MahmoodLab UNI  ViT-L         (GATED: HF token+access, 1024-d, 224px)
#   uni2     : MahmoodLab UNI2-h ViT-H       (GATED, 1536-d, 224px)
#   gigapath : Prov-GigaPath ViT-g           (GATED, 1536-d, 224px; large)
ENCODER  = "phikon"   # <-- default: open pathology FM, no token. (options: dinov2/phikon/phikon2/uni/uni2/gigapath)
HF_TOKEN = ""          # gated models: paste 'hf_...' OR add Kaggle secret HF_TOKEN
_ENC_REG = {"dinov2": (768, 518), "phikon": (768, 224), "phikon2": (1024, 224),
            "uni": (1024, 224), "uni2": (1536, 224), "gigapath": (1536, 224)}

def _hf_login():
    """Authenticate to HuggingFace for gated encoders (token / Kaggle secret)."""
    tok = HF_TOKEN or os.environ.get("HF_TOKEN", "")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = ""
    if tok:
        try:
            from huggingface_hub import login; login(token=tok); return True
        except Exception as e:
            print("HF login failed:", e)
    return False

class _HFWrap(nn.Module):
    """Wrap a HuggingFace ViT so forward(x) returns the CLS embedding (B, D)."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(pixel_values=x).last_hidden_state[:, 0]

def build_backbone(enc):
    """Return (backbone_module, embed_dim); backbone(x) -> (B, D) CLS embeddings."""
    if enc == "dinov2":
        return torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_reg"), 768
    if enc in ("phikon", "phikon2"):
        from transformers import AutoModel
        repo = "owkin/phikon" if enc == "phikon" else "owkin/phikon-v2"
        return _HFWrap(AutoModel.from_pretrained(repo)), _ENC_REG[enc][0]
    if enc in ("uni", "uni2"):
        import timm; _hf_login()
        repo = "MahmoodLab/UNI" if enc == "uni" else "MahmoodLab/UNI2-h"
        return timm.create_model(f"hf-hub:{repo}", pretrained=True,
                                 init_values=1e-5, dynamic_img_size=True), _ENC_REG[enc][0]
    if enc == "gigapath":
        import timm; _hf_login()
        return timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True), _ENC_REG[enc][0]
    raise ValueError("unknown ENCODER: " + enc)

# validate the chosen encoder NOW (fail fast, before the expensive stages 2-5)
try:
    _bb, _d = build_backbone(ENCODER)
    del _bb; gc.collect(); torch.cuda.empty_cache()
    print(f"Encoder '{ENCODER}' loaded OK (dim={_d}).")
except Exception as e:
    print("\n" + "!" * 64)
    print(f"!! Could not load encoder '{ENCODER}': {e}")
    print("!! GATED models (uni/uni2/gigapath) require HuggingFace access + token:")
    print("!!   1) Request access on the model's HF page (e.g. huggingface.co/MahmoodLab/UNI)")
    print("!!   2) Add-ons -> Secrets -> add 'HF_TOKEN', or set HF_TOKEN='hf_...' above.")
    print("!! Open alternatives needing NO token: ENCODER='phikon' or 'phikon2'.")
    print("!! Falling back to DINOv2 baseline so the run still completes.")
    print("!" * 64 + "\n")
    ENCODER = "dinov2"

EMBED_DIM, INPUT_SIZE = _ENC_REG[ENCODER]
NODE_FEAT_DIM = EMBED_DIM + 3            # embedding + scale flag + 2 coords
# Fine-tune the last FT_LAST_BLOCKS of the encoder for the patch classifiers
# (closes the binary/GP gap vs the base paper, which fine-tuned its backbone).
# UNI/GigaPath (ViT-L/g) kept frozen to fit T4 memory; flip here if on a bigger GPU.
FINETUNE_BACKBONE = ENCODER in ("dinov2", "phikon", "phikon2")
print(f"ENCODER={ENCODER} | EMBED_DIM={EMBED_DIM} | INPUT_SIZE={INPUT_SIZE} "
      f"| NODE_FEAT_DIM={NODE_FEAT_DIM} | finetune_backbone={FINETUNE_BACKBONE}")

print("Stage 1 complete: seeds set, paths and hyperparameters defined.")

PyTorch: 2.10.0+cu128
torchvision: 0.25.0+cu128
torch-geometric: 2.8.0
Device: cuda
>>> FULL run mode <<<  (PANDA external test: 300 RANDOM slides, 384 tiles/slide)


config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Encoder 'phikon' loaded OK (dim=768).
ENCODER=phikon | EMBED_DIM=768 | INPUT_SIZE=224 | NODE_FEAT_DIM=771 | finetune_backbone=True
Stage 1 complete: seeds set, paths and hyperparameters defined.


## Stage 2 — SICAPv2 patch labels from masks

In [4]:
import cv2, pandas as pd
from tqdm.auto import tqdm

def tissue_fraction(bgr):
    """Fraction of non-background tissue via Otsu on the grayscale image."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return float((th > 0).mean())

# SICAPv2 integer-label encoding (0 = background). If the unique values printed
# by Stage 2 disagree with this, edit the map here. Typical: 0=bg,1=Benign,2=GP3,3=GP4,4=GP5.
MASK_INT_MAP = {1: "Benign", 2: "GP3", 3: "GP4", 4: "GP5"}

def find_mask(msk_dir, fn):
    """Locate a patch's mask, tolerant to extension / _mask suffix differences."""
    stem = Path(fn).stem
    for c in (fn, stem + ".png", stem + ".jpg", stem + ".jpeg", stem + ".tif",
              stem + ".bmp", stem + "_mask.png", stem + "_mask.jpg"):
        p = Path(msk_dir) / c
        if p.exists():
            return p
    return None

def detect_mask_encoding(msk_dir, files):
    """Decide if masks are RGB colour-coded or single-channel integer labels."""
    vals, rgb, checked = set(), False, 0
    for fn in files:
        p = find_mask(msk_dir, fn)
        if p is None:
            continue
        m = cv2.imread(str(p))
        if m is None:
            continue
        if not (np.array_equal(m[..., 0], m[..., 1]) and np.array_equal(m[..., 1], m[..., 2])):
            rgb = True
        vals.update(np.unique(m[..., 0]).tolist())
        checked += 1
        if checked >= 15:
            break
    mode = "rgb" if rgb else "int"
    print(f"  mask encoding: {mode.upper()} | sample unique pixel values: {sorted(vals)[:12]}")
    if mode == "int":
        print(f"  integer-label map in use: {MASK_INT_MAP}  (0 treated as background)")
    return mode

def mask_to_label(mask, mode):
    """Dominant class from a SICAP mask. mode='int' (label index) or 'rgb' (colour)."""
    if mode == "int":
        m = mask[..., 0] if mask.ndim == 3 else mask
        counts = {"Benign": 0, "GP3": 0, "GP4": 0, "GP5": 0}
        for v, name in MASK_INT_MAP.items():
            counts[name] += int((m == v).sum())
        if sum(counts.values()) == 0:
            return None, counts
        return max(counts, key=counts.get), counts
    R, G, B = mask[..., 0], mask[..., 1], mask[..., 2]
    bg   = (R > 200) & (G > 200) & (B > 200)
    gp5  = (R > 200) & (G < 100) & (B < 100)          # red
    gp4  = (R < 100) & (G < 100) & (B > 200)          # blue
    gp3  = (R < 100) & (G > 200) & (B < 100)          # green
    benign = (~bg) & (~gp5) & (~gp4) & (~gp3)          # other non-white
    counts = {"Benign": int(benign.sum()), "GP3": int(gp3.sum()),
              "GP4": int(gp4.sum()), "GP5": int(gp5.sum())}
    if sum(counts.values()) == 0:
        return None, counts
    return max(counts, key=counts.get), counts

def build_label_csv(img_dir, msk_dir, out_csv, cap=None):
    """Scan an Images/Masks pair and write per-patch labels to out_csv."""
    rows = []
    if not Path(img_dir).exists():
        print("MISSING:", img_dir); pd.DataFrame().to_csv(out_csv, index=False); return out_csv
    files = sorted([f for f in os.listdir(img_dir)
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))])
    if cap and len(files) > cap:
        # SLIDE-AWARE smoke sample: keep ALL patches of a few random slides so the
        # smoke run produces realistic multi-patch graphs (not 1 patch/slide).
        from collections import defaultdict
        by_slide = defaultdict(list)
        for f in files:
            by_slide[get_slide_id(f)].append(f)
        sids = list(by_slide.keys())
        np.random.RandomState(SEED).shuffle(sids)
        picked = []
        for s in sids:
            picked += by_slide[s]
            if len(picked) >= cap:
                break
        files = picked
    mode = detect_mask_encoding(msk_dir, files)
    skip = {"no_mask": 0, "bad_img": 0, "low_tissue": 0, "bad_mask": 0, "no_fg": 0}
    n_masks_dir = len([f for f in os.listdir(msk_dir)]) if Path(msk_dir).exists() else 0
    for fn in tqdm(files, desc=Path(out_csv).name):
        ip = Path(img_dir) / fn
        mp = find_mask(msk_dir, fn)
        if mp is None:
            skip["no_mask"] += 1; continue
        bgr = cv2.imread(str(ip))
        if bgr is None:
            skip["bad_img"] += 1; continue
        tf = tissue_fraction(bgr)
        if tf < TISSUE_THRESHOLD:
            skip["low_tissue"] += 1; continue
        mask = cv2.imread(str(mp))
        if mask is None:
            skip["bad_mask"] += 1; continue
        label, _ = mask_to_label(cv2.cvtColor(mask, cv2.COLOR_BGR2RGB), mode)
        if label is None:
            skip["no_fg"] += 1; continue
        rows.append({"filename": fn, "label": label, "label_int": LABEL2INT[label],
                     "tissue_fraction": round(tf, 4), "src_dir": str(img_dir),
                     "binary_label": 0 if label == "Benign" else 1})
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"  [{Path(out_csv).name}] images={len(files)} masks_in_dir={n_masks_dir} "
          f"kept={len(df)} | skipped: {skip}")
    return out_csv

def build_label_csv_partition(out_csv, cap=None):
    """Clean SICAP patch labels from the OFFICIAL partition (Train.xlsx+Test.xlsx,
    one-hot NC/G3/G4/G5). Ground truth -> fixes the noisy mask-derived labels."""
    frames = []
    for nm in ("Train.xlsx", "Test.xlsx"):
        fp = SICAP_PARTITION / nm
        if fp.exists():
            frames.append(pd.read_excel(fp))
    if not frames:
        print("  partition files not found -> falling back to masks.")
        return build_label_csv(SICAP_TR_IMG, SICAP_TR_MSK, out_csv, cap)
    lab = pd.concat(frames, ignore_index=True).drop_duplicates("image_name").reset_index(drop=True)
    names = ["Benign", "GP3", "GP4", "GP5"]
    lab["label"] = [names[i] for i in lab[["NC", "G3", "G4", "G5"]].values.argmax(1)]
    lab["slide_id"] = lab["image_name"].apply(get_slide_id)
    if cap and len(lab) > cap:                       # slide-aware smoke subset
        sids = lab["slide_id"].drop_duplicates().tolist()
        np.random.RandomState(SEED).shuffle(sids)
        chosen, picked = set(), lab.iloc[0:0]
        for s in sids:
            chosen.add(s); picked = lab[lab["slide_id"].isin(chosen)]
            if len(picked) >= cap:
                break
        lab = picked
    rows, miss = [], 0
    for _, r in tqdm(lab.iterrows(), total=len(lab), desc="partition labels"):
        fn = r["image_name"]
        if not (SICAP_TR_IMG / fn).exists():
            miss += 1; continue
        rows.append({"filename": fn, "label": r["label"], "label_int": LABEL2INT[r["label"]],
                     "tissue_fraction": 1.0, "src_dir": str(SICAP_TR_IMG),
                     "binary_label": 0 if r["label"] == "Benign" else 1})
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False)
    print(f"  [partition] {len(lab)} labelled rows -> {len(df)} patches ({miss} images missing) "
          f"| dist: {df['label'].value_counts().to_dict() if len(df) else {}}")
    return out_csv

# ---- Gleason2019 ingestion (Route A: extra TRAINING cohort) -----------------
_IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def _g19_list_images():
    """Recursively list Gleason2019 core images (handles nested folders)."""
    out = []
    if GLEASON_IMG.exists():
        for dp, _, fs in os.walk(GLEASON_IMG):
            out += [os.path.join(dp, f) for f in fs if f.lower().endswith(_IMG_EXT)]
    return sorted(out)

def _g19_mask_index():
    """Recursively index STAPLE masks by filename stem (handles nesting)."""
    idx = {}
    if GLEASON_MASK.exists():
        for dp, _, fs in os.walk(GLEASON_MASK):
            for f in fs:
                if f.lower().endswith(_IMG_EXT):
                    idx[Path(f).stem.lower()] = os.path.join(dp, f)
    return idx

def _g19_find_mask(stem, idx):
    """Match a core to its STAPLE mask. Gleason2019 masks abbreviate the core id:
    'slide001_core003' (image) -> 's001_c003' (mask)."""
    key = stem.lower()
    if key in idx:
        return idx[key]
    m = re.search(r"slide(\d+).*?core(\d+)", key)   # slideNNN_coreMMM -> sNNN_cMMM
    if m:
        alt = f"s{m.group(1)}_c{m.group(2)}"
        if alt in idx:
            return idx[alt]
    for k, v in idx.items():                        # last-resort substring match
        if key in k or k in key:
            return v
    return None

def _g19_label(mask_region):
    """Dominant Gleason class of a mask tile via GLEASON_MASK_MAP (None if empty)."""
    m = mask_region[..., 0] if mask_region.ndim == 3 else mask_region
    counts = {"Benign": 0, "GP3": 0, "GP4": 0, "GP5": 0}
    for v, name in GLEASON_MASK_MAP.items():
        counts[name] += int((m == v).sum())
    if sum(counts.values()) == 0:
        return None
    return max(counts, key=counts.get)

def ingest_gleason2019(out_dir, cap_cores=None, cap_tiles=80):
    """Tile Gleason2019 TMA cores into 512px patches labelled from the STAPLE
    consensus masks; saves tiles to out_dir and returns a SICAP-format DataFrame."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cores = _g19_list_images()
    midx = _g19_mask_index()
    print(f"  Gleason2019: found {len(cores)} core images, {len(midx)} masks "
          f"(img dir exists={GLEASON_IMG.exists()}, mask dir exists={GLEASON_MASK.exists()})")
    if not cores:
        print("  -> no Gleason2019 cores found; training stays SICAP-only.")
        return pd.DataFrame()
    if cap_cores:
        cores = cores[:cap_cores]
    rows, uniq, used, nomask = [], set(), 0, 0
    for ci, fp in enumerate(tqdm(cores, desc="gleason2019 ingest")):
        stem = Path(fp).stem
        mp = _g19_find_mask(stem, midx)
        if mp is None:
            nomask += 1; continue
        img = cv2.imread(fp); mask = cv2.imread(str(mp))
        if img is None or mask is None:
            continue
        if mask.shape[:2] != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        if ci < 8:
            uniq.update(np.unique(mask[..., 0]).tolist())
        H, W = img.shape[:2]
        sid = "G19" + re.sub(r"[^A-Za-z0-9]", "", stem)   # one token -> get_slide_id groups by core
        kept = 0
        for yy in range(0, H - GLEASON_TILE + 1, GLEASON_TILE):
            if kept >= cap_tiles:
                break
            for xx in range(0, W - GLEASON_TILE + 1, GLEASON_TILE):
                tile = img[yy:yy + GLEASON_TILE, xx:xx + GLEASON_TILE]
                if tissue_fraction(tile) < TISSUE_THRESHOLD:
                    continue
                lab = _g19_label(mask[yy:yy + GLEASON_TILE, xx:xx + GLEASON_TILE])
                if lab is None:
                    continue
                tfn = f"{sid}_{yy // GLEASON_TILE}_{xx // GLEASON_TILE}.jpg"
                cv2.imwrite(str(out_dir / tfn), tile)
                rows.append({"filename": tfn, "label": lab, "label_int": LABEL2INT[lab],
                             "tissue_fraction": 1.0, "src_dir": str(out_dir),
                             "binary_label": 0 if lab == "Benign" else 1})
                kept += 1
                if kept >= cap_tiles:
                    break
        if kept:
            used += 1
    df = pd.DataFrame(rows)
    print(f"  Gleason2019: {used}/{len(cores)} cores tiled -> {len(df)} patches "
          f"({nomask} cores had no matching mask) | mask unique values: {sorted(uniq)[:12]}")
    if len(df):
        print("  Gleason2019 class dist:", df["label"].value_counts().to_dict())
    return df

# The provided SICAPv2 Val/ folder holds only ~18 placeholder images, so we IGNORE
# it and build a proper SLIDE-LEVEL validation split from the large Train/ set.
# Splitting by slide_id guarantees no patch/slide leakage between train and val.
from sklearn.model_selection import train_test_split

if USE_PARTITION_LABELS:
    ALL_CSV = build_label_csv_partition(WORK / "sicap_all_labels.csv", MAX_TRAIN_PATCHES)
else:
    ALL_CSV = build_label_csv(SICAP_TR_IMG, SICAP_TR_MSK, WORK / "sicap_all_labels.csv", MAX_TRAIN_PATCHES)
df_all = pd.read_csv(ALL_CSV)
TRAIN_CSV = WORK / "sicap_train_labels.csv"
VAL_CSV   = WORK / "sicap_val_labels.csv"

if len(df_all) == 0:
    df_all.to_csv(TRAIN_CSV, index=False); df_all.to_csv(VAL_CSV, index=False)
    print("WARNING: no labelled patches found.")
else:
    df_all["slide_id"] = df_all["filename"].apply(get_slide_id)
    _pps = df_all.groupby("slide_id").size()
    print("  example filenames:", df_all["filename"].head(5).tolist())
    print(f"  slides={_pps.shape[0]} | patches/slide: mean={_pps.mean():.1f} "
          f"max={int(_pps.max())} min={int(_pps.min())}"
          + ("   <-- WARNING: ~1 patch/slide means get_slide_id() needs fixing"
             if _pps.mean() < 2 else ""))

    # ---- ground-truth slide labels + patient ids from wsi_labels.xlsx ----
    def gleason_to_isup(p, s):
        """Map Gleason primary+secondary to ISUP grade (0 = benign/non-cancer)."""
        try:
            p, s = int(p), int(s)
        except Exception:
            return 0
        if p < 3 or s < 3:
            return 0
        t = p + s
        if t <= 6:          return 1
        if p == 3 and s == 4: return 2
        if p == 4 and s == 3: return 3
        if t == 8:          return 4
        return 5
    SID2PATIENT, SID2ISUP = {}, {}
    try:
        _w = pd.read_excel(SICAP_WSI_LABELS)
        _w["slide_id"] = _w["slide_id"].astype(str).str.strip()
        _w["isup_grade"] = _w.apply(lambda r: gleason_to_isup(r["Gleason_primary"],
                                                              r["Gleason_secondary"]), axis=1)
        SID2PATIENT = dict(zip(_w["slide_id"], _w["patient_id"].astype(str)))
        SID2ISUP = dict(zip(_w["slide_id"], _w["isup_grade"]))
        match = df_all["slide_id"].isin(SID2PATIENT).mean()
        print(f"  wsi_labels: {len(_w)} slides | filename->wsi match rate: {match:.0%} "
              + ("(GOOD)" if match > 0.8 else "(LOW - check slide_id format!)"))
    except Exception as e:
        print("  (wsi_labels not used, falling back to mask-derived labels:", e, ")")

    # patient id per patch (fall back to slide_id when not in wsi_labels)
    df_all["patient"] = df_all["slide_id"].map(lambda s: SID2PATIENT.get(s, s))

    # ---- PATIENT-level stratified split (prevents patient leakage) ----
    grp = df_all.groupby("patient")["label_int"].agg(lambda s: s.value_counts().index[0]).reset_index()
    can_strat = grp["label_int"].nunique() > 1 and grp["label_int"].value_counts().min() >= 2
    if len(grp) >= 5:
        tr_p, va_p = train_test_split(grp["patient"], test_size=0.2, random_state=SEED,
                                      stratify=grp["label_int"] if can_strat else None)
        tr_set, va_set = set(tr_p), set(va_p)
    else:
        print("  WARNING: <5 patients -> train/val overlap (smoke only).")
        tr_set = va_set = set(grp["patient"])
    sicap_tr = df_all[df_all["patient"].isin(tr_set)].copy()
    sicap_va = df_all[df_all["patient"].isin(va_set)].copy()
    sicap_va.to_csv(VAL_CSV, index=False)   # validation stays PURE SICAP (clean internal metric)
    print(f"  patient-level split: {len(tr_set)} train / {len(va_set)} val patients "
          f"| {df_all['slide_id'].nunique()} slides, {len(df_all)} patches")

    # add Gleason2019 cores to TRAINING only (Route A: cross-cohort diversity)
    train_df = sicap_tr
    if USE_GLEASON:
        df_g19 = ingest_gleason2019(WORK / "gleason_tiles", GLEASON_CAP_CORES, GLEASON_CAP_TILES)
        if len(df_g19):
            df_g19["slide_id"] = df_g19["filename"].apply(get_slide_id)
            df_g19["patient"] = df_g19["slide_id"]   # each core = its own patient
            train_df = pd.concat([sicap_tr, df_g19], ignore_index=True)
    train_df.to_csv(TRAIN_CSV, index=False)
    print(f"  TRAIN total: {len(train_df)} patches "
          f"({len(sicap_tr)} SICAP + {len(train_df) - len(sicap_tr)} Gleason2019) "
          f"| {train_df['slide_id'].nunique()} train slides")

for name, p in [("TRAIN", TRAIN_CSV), ("VAL", VAL_CSV)]:
    df = pd.read_csv(p)
    print(f"\n{name} ({len(df)} patches) class distribution:")
    if len(df):
        print(df["label"].value_counts())

# peek at slide-level ground-truth labels (used to validate our >5% rule later)
try:
    _wl = pd.read_excel(SICAP_WSI_LABELS)
    print("\nwsi_labels.xlsx columns:", list(_wl.columns))
    print(_wl.head(4).to_string())
except Exception as e:
    print("\n(could not read wsi_labels.xlsx:", e, ")")
print("\nStage 2 complete.")

partition labels:   0%|          | 0/12081 [00:00<?, ?it/s]

  [partition] 12081 labelled rows -> 12081 patches (0 images missing) | dist: {'GP4': 4494, 'Benign': 4417, 'GP3': 2222, 'GP5': 948}
  example filenames: ['16B0001851_Block_Region_1_0_0_xini_6803_yini_59786.jpg', '16B0001851_Block_Region_1_0_1_xini_7827_yini_59786.jpg', '16B0001851_Block_Region_1_0_2_xini_8851_yini_59786.jpg', '16B0001851_Block_Region_1_0_3_xini_9875_yini_59786.jpg', '16B0001851_Block_Region_1_1_0_xini_6803_yini_60810.jpg']
  slides=155 | patches/slide: mean=77.9 max=347 min=7
  wsi_labels: 155 slides | filename->wsi match rate: 100% (GOOD)
  patient-level split: 76 train / 19 val patients | 155 slides, 12081 patches
  Gleason2019: found 244 core images, 245 masks (img dir exists=True, mask dir exists=True)


gleason2019 ingest:   0%|          | 0/244 [00:00<?, ?it/s]

  Gleason2019: 244/244 cores tiled -> 14526 patches (0 cores had no matching mask) | mask unique values: [0, 1, 3, 4, 5]
  Gleason2019 class dist: {'Benign': 6675, 'GP4': 4586, 'GP3': 3110, 'GP5': 155}
  TRAIN total: 24447 patches (9921 SICAP + 14526 Gleason2019) | 366 train slides

TRAIN (24447 patches) class distribution:
label
Benign    10033
GP4        8360
GP3        5095
GP5         959
Name: count, dtype: int64

VAL (2160 patches) class distribution:
label
Benign    1059
GP4        720
GP3        237
GP5        144
Name: count, dtype: int64

wsi_labels.xlsx columns: ['slide_id', 'patient_id', 'Gleason_primary', 'Gleason_secondary']
     slide_id  patient_id  Gleason_primary  Gleason_secondary
0  16B0001851      667360                4                  5
1  16B0003388      325687                4                  4
2  16B0003394      747184                3                  3
3  16B0006668       14107                5                  5

Stage 2 complete.


## Stage 3 — Stain normalisation (Macenko)

Tries `staintools`; falls back to a self-contained numpy Macenko normaliser so
the pipeline never blocks on the spams build.

In [5]:
from PIL import Image
import matplotlib.pyplot as plt

class _MacenkoFallback:
    """Pure-numpy Macenko stain normaliser (used if staintools is unavailable)."""
    def __init__(self, Io=240, alpha=1, beta=0.15):
        self.Io, self.alpha, self.beta = Io, alpha, beta
        self.HERef = np.array([[0.5626, 0.2159], [0.7201, 0.8012], [0.4062, 0.5581]])
        self.maxCRef = np.array([1.9705, 1.0308])

    def _stats(self, img):
        img = img.reshape(-1, 3).astype(float)
        OD = -np.log((img + 1) / self.Io)
        ODhat = OD[~np.any(OD < self.beta, axis=1)]
        if ODhat.shape[0] < 10:
            raise ValueError("not enough tissue")
        _, V = np.linalg.eigh(np.cov(ODhat.T))
        V = V[:, [2, 1]] if V.shape[1] == 3 else V[:, -2:]
        That = ODhat.dot(V)
        phi = np.arctan2(That[:, 1], That[:, 0])
        minP, maxP = np.percentile(phi, self.alpha), np.percentile(phi, 100 - self.alpha)
        v1 = V.dot(np.array([np.cos(minP), np.sin(minP)]))
        v2 = V.dot(np.array([np.cos(maxP), np.sin(maxP)]))
        HE = np.array([v1, v2]).T if v1[0] > v2[0] else np.array([v2, v1]).T
        Y = OD.reshape(-1, 3).T
        C = np.linalg.lstsq(HE, Y, rcond=None)[0]
        maxC = np.array([np.percentile(C[0], 99), np.percentile(C[1], 99)])
        return HE, C, maxC

    def fit(self, target):
        self.HERef_src, _, self.maxCRef = self._stats(target)

    def transform(self, img):
        h, w, _ = img.shape
        HE, C, maxC = self._stats(img)
        C = C / maxC[:, None] * self.maxCRef[:, None]
        out = self.Io * np.exp(-self.HERef.dot(C))
        out = np.clip(out, 0, 255).T.reshape(h, w, 3).astype(np.uint8)
        return out

# build normaliser (staintools if importable, else fallback)
USE_STAINTOOLS = False
try:
    import staintools
    _norm = staintools.StainNormalizer(method="macenko")
    USE_STAINTOOLS = True
    print("Using staintools Macenko normaliser.")
except Exception as e:
    _norm = _MacenkoFallback()
    print("staintools unavailable -> numpy Macenko fallback:", e)

def _first_valid_target(csv_path, img_dir):
    df = pd.read_csv(csv_path)
    for fn in df["filename"]:
        arr = cv2.imread(str(Path(img_dir) / fn))
        if arr is not None:
            return cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
    raise RuntimeError("no valid target patch found")

TARGET = _first_valid_target(TRAIN_CSV, SICAP_TR_IMG)
try:
    if USE_STAINTOOLS:
        std = staintools.LuminosityStandardizer.standardize(TARGET)
        _norm.fit(std)
    else:
        _norm.fit(TARGET)
except Exception as e:
    print("Normaliser fit failed, will pass-through:", e)

def normalise_patch(rgb):
    """Macenko-normalise an RGB uint8 array; return original on any failure."""
    try:
        if USE_STAINTOOLS:
            return _norm.transform(staintools.LuminosityStandardizer.standardize(rgb))
        return _norm.transform(rgb)
    except Exception:
        return rgb

def normalise_split(csv_path, out_dir):
    """Write Macenko-normalised copies of every patch (sourced from its src_dir)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(csv_path)
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"norm {out_dir.name}"):
        fn = r["filename"]; op = out_dir / fn
        if op.exists():
            continue
        arr = cv2.imread(str(Path(r["src_dir"]) / fn))
        if arr is None:
            continue
        rgb = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
        Image.fromarray(normalise_patch(rgb)).save(op)
    return out_dir

NORM_TR = normalise_split(TRAIN_CSV, WORK / "sicap_norm/train")
NORM_VA = normalise_split(VAL_CSV,   WORK / "sicap_norm/val")

# visual check
df = pd.read_csv(TRAIN_CSV)
ex = df.head(3)
if len(ex):
    fig, ax = plt.subplots(len(ex), 2, figsize=(6, 3 * len(ex)))
    ax = np.atleast_2d(ax)
    for i, (_, r) in enumerate(ex.iterrows()):
        fn = r["filename"]
        o = cv2.cvtColor(cv2.imread(str(Path(r["src_dir"]) / fn)), cv2.COLOR_BGR2RGB)
        n = np.array(Image.open(NORM_TR / fn))
        ax[i, 0].imshow(o); ax[i, 0].set_title("orig"); ax[i, 0].axis("off")
        ax[i, 1].imshow(n); ax[i, 1].set_title("norm"); ax[i, 1].axis("off")
    plt.tight_layout(); plt.savefig(WORK / "stain_examples.png", dpi=80); plt.close()
print("Stage 3 complete.")

Using staintools Macenko normaliser.


norm train:   0%|          | 0/24447 [00:00<?, ?it/s]

norm val:   0%|          | 0/2160 [00:00<?, ?it/s]

Stage 3 complete.


## Stage 4 — Multi-scale patch generation (10x full / 20x centre-crop)

In [6]:
def make_multiscale(norm_dir, split):
    """Create 10x (full->518) and 20x (centre256->518) versions of each patch."""
    base10 = WORK / f"sicap_multiscale/{split}/10x"
    base20 = WORK / f"sicap_multiscale/{split}/20x"
    base10.mkdir(parents=True, exist_ok=True); base20.mkdir(parents=True, exist_ok=True)
    files = sorted(os.listdir(norm_dir))
    for fn in tqdm(files, desc=f"multiscale {split}"):
        img = Image.open(Path(norm_dir) / fn).convert("RGB")
        img.resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC).save(base10 / fn)
        c = (PATCH_SIZE - 256) // 2
        img.crop((c, c, c + 256, c + 256)).resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC).save(base20 / fn)
    return base10, base20

def add_paths(csv_path, b10, b20):
    df = pd.read_csv(csv_path)
    df["patch_path_10x"] = df["filename"].apply(lambda f: str(Path(b10) / f))
    df["patch_path_20x"] = df["filename"].apply(lambda f: str(Path(b20) / f))
    keep = df[df["patch_path_10x"].apply(os.path.exists)].reset_index(drop=True)
    keep.to_csv(csv_path, index=False)
    return keep

b10_tr, b20_tr = make_multiscale(NORM_TR, "train")
b10_va, b20_va = make_multiscale(NORM_VA, "val")
dtr = add_paths(TRAIN_CSV, b10_tr, b20_tr)
dva = add_paths(VAL_CSV,   b10_va, b20_va)
print(f"Train 10x/20x patches: {len(dtr)} | Val: {len(dva)}")
print("Stage 4 complete.")

multiscale train:   0%|          | 0/24447 [00:00<?, ?it/s]

multiscale val:   0%|          | 0/2160 [00:00<?, ?it/s]

Train 10x/20x patches: 24447 | Val: 2160
Stage 4 complete.


## Stage 5 — Patch Dataset / DataLoader (weighted sampler)

In [7]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def train_tf():
    """Augmentation pipeline for patch training."""
    return T.Compose([
        T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.5),
        T.RandomChoice([T.RandomRotation((d, d)) for d in (0, 90, 180, 270)]),
        T.ColorJitter(0.4, 0.4, 0.3, 0.1),   # heavier stain jitter -> cross-scanner robustness
        T.RandomApply([T.GaussianBlur(3)], p=0.3),
        T.ToTensor(), T.Normalize(MEAN, STD)])

def eval_tf():
    """Deterministic transform for val / feature extraction."""
    return T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

class SICAPPatchDataset(Dataset):
    """Loads patches at a given scale; label column is configurable."""
    def __init__(self, csv_path, scale="10x", split="train", transform=None, label_col="label_int"):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.col = f"patch_path_{scale}"
        self.transform = transform or eval_tf()
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r[self.col]).convert("RGB")
        return self.transform(img), int(r[self.label_col])

def make_sampler(csv_path, label_col="label_int"):
    """WeightedRandomSampler with inverse-frequency class weights (None if empty)."""
    df = pd.read_csv(csv_path)
    labels = df[label_col].dropna().astype(int).values
    if len(labels) == 0:
        return None
    counts = np.bincount(labels, minlength=int(labels.max()) + 1)
    w_class = 1.0 / np.clip(counts, 1, None)
    weights = w_class[labels]
    return WeightedRandomSampler(torch.DoubleTensor(weights), len(weights), replacement=True)

def make_loaders(scale="10x", label_col="label_int", train_csv=None, val_csv=None):
    train_csv = train_csv or TRAIN_CSV; val_csv = val_csv or VAL_CSV
    tr = SICAPPatchDataset(train_csv, scale, "train", train_tf(), label_col)
    va = SICAPPatchDataset(val_csv, scale, "val", eval_tf(), label_col)
    samp = make_sampler(train_csv, label_col)
    tl = DataLoader(tr, batch_size=BATCH_SIZE_PATCH, sampler=samp,
                    shuffle=(samp is None and len(tr) > 0),   # avoid RandomSampler on empty set
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
    vl = DataLoader(va, batch_size=BATCH_SIZE_PATCH, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    return tl, vl

_tl, _vl = make_loaders("10x", "binary_label")
xb, yb = next(iter(_tl))
print("Batch:", xb.shape, yb.shape)
print("Stage 5 complete.")

Batch: torch.Size([16, 3, 224, 224]) torch.Size([16])
Stage 5 complete.


## Stage 6 — DINOv2 binary patch classifier (frozen head -> fine-tune)

In [8]:
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, f1_score

def load_dinov2(num_classes):
    """Build {selected encoder backbone -> (B,D)} + a linear head (per ENCODER)."""
    backbone, dim = build_backbone(ENCODER)
    model = nn.Module()
    model.backbone = backbone
    model.head = nn.Linear(dim, num_classes)
    def forward(x):
        return model.head(model.backbone(x))   # CLS embedding -> logits
    model.forward = forward
    return model.to(DEVICE)

def set_backbone_trainable(model, flag, last_k=None):
    """Freeze/unfreeze backbone. If last_k is set, unfreeze only the last K
    transformer blocks (+ final norm). Handles DINOv2/timm (.blocks/.norm) AND
    HuggingFace ViT (phikon: .m.encoder.layer / .m.layernorm)."""
    bb = model.backbone
    for p in bb.parameters():
        p.requires_grad = (flag and last_k is None)
    if flag and last_k is not None:
        blocks, norm = None, None
        if hasattr(bb, "blocks"):                                   # dinov2 / timm ViT
            blocks, norm = list(bb.blocks), getattr(bb, "norm", None)
        elif hasattr(bb, "m") and hasattr(getattr(bb.m, "encoder", None), "layer"):  # HF ViT (phikon)
            blocks, norm = list(bb.m.encoder.layer), getattr(bb.m, "layernorm", None)
        if blocks:
            for blk in blocks[-last_k:]:
                for p in blk.parameters():
                    p.requires_grad = True
        if norm is not None:
            for p in norm.parameters():
                p.requires_grad = True

@torch.no_grad()
def evaluate(model, loader, num_classes):
    """Return (loss, acc, auroc-or-qwk dict) on a loader."""
    model.eval(); losses, ys, ps, logits_all = [], [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = model(x); loss = F.cross_entropy(out, y)
        losses.append(loss.item())
        ys.append(y.cpu().numpy())
        ps.append(out.argmax(1).cpu().numpy())
        logits_all.append(torch.softmax(out.float(), 1).cpu().numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps); pr = np.concatenate(logits_all)
    acc = accuracy_score(ys, ps)
    metrics = {"acc": acc, "qwk": cohen_kappa_score(ys, ps, weights="quadratic") if len(set(ys)) > 1 else 0.0}
    try:
        if num_classes == 2:
            metrics["auroc"] = roc_auc_score(ys, pr[:, 1])
        else:
            metrics["auroc"] = roc_auc_score(ys, pr, multi_class="ovr", average="macro")
    except Exception:
        metrics["auroc"] = float("nan")
    return float(np.mean(losses)), metrics

def train_classifier(model, train_csv, val_csv, num_classes, label_col, ckpt,
                     monitor="loss", scale="10x"):
    """Two-phase training: frozen head then full fine-tune; early stops; saves best."""
    tl, vl = make_loaders(scale, label_col, train_csv, val_csv)
    if len(tl.dataset) == 0 or len(vl.dataset) == 0:
        print(f"  SKIP training: empty subset (train={len(tl.dataset)}, val={len(vl.dataset)}). "
              f"Saving an untrained checkpoint so later stages can still load it.")
        torch.save(model.state_dict(), ckpt)
        return model
    torch.save(model.state_dict(), ckpt)   # ensure ckpt ALWAYS exists (e.g. NaN-AUROC val in smoke)
    scaler = torch.cuda.amp.GradScaler()
    best, best_state, bad = (1e9 if monitor == "loss" else -1e9), None, 0

    def run_phase(epochs, opt, sched=None, tag=""):
        nonlocal best, best_state, bad
        for ep in range(epochs):
            model.train(); tr_losses = []
            for x, y in tqdm(tl, desc=f"{tag} ep{ep}", leave=False):
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                with torch.cuda.amp.autocast():
                    loss = F.cross_entropy(model(x), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                if sched is not None:
                    sched.step()
                tr_losses.append(loss.item())
            vloss, vm = evaluate(model, vl, num_classes)
            score = -vloss if monitor == "loss" else vm[monitor]
            improved = score > (-best if monitor == "loss" else best)
            print(f"[{tag}] ep{ep} train_loss={np.mean(tr_losses):.4f} "
                  f"val_loss={vloss:.4f} val_acc={vm['acc']:.4f} "
                  f"val_auroc={vm.get('auroc', float('nan')):.4f} val_qwk={vm['qwk']:.4f}")
            cur = vloss if monitor == "loss" else vm[monitor]
            better = cur < best if monitor == "loss" else cur > best
            if better:
                best, best_state, bad = cur, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
                torch.save(model.state_dict(), ckpt)
            else:
                bad += 1
                if bad >= PATIENCE:
                    print(f"  early stop ({tag})"); return

    set_backbone_trainable(model, False)
    opt = torch.optim.AdamW(model.head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    run_phase(EPOCHS_FROZEN, opt, None, "frozen")

    if FINETUNE_BACKBONE:
        # DINOv2: partial fine-tune of the last K blocks. Pathology FMs are kept
        # frozen (linear probe) — standard practice, and avoids huge-ViT OOM.
        torch.cuda.empty_cache()
        set_backbone_trainable(model, True, last_k=FT_LAST_BLOCKS)
        trainable = [p for p in model.parameters() if p.requires_grad]
        print(f"  fine-tuning last {FT_LAST_BLOCKS} blocks: "
              f"{sum(p.numel() for p in trainable)/1e6:.1f}M trainable params")
        opt = torch.optim.AdamW(trainable, lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
        steps = max(1, len(tl) * EPOCHS_UNFREEZE)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR_FINETUNE, total_steps=steps, pct_start=0.3)
        run_phase(EPOCHS_UNFREEZE, opt, sched, "finetune")
    else:
        print(f"  ENCODER='{ENCODER}': frozen feature extractor (linear probe only).")
    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        torch.save(model.state_dict(), ckpt)   # never "improved" (e.g. NaN metric) -> save trained final
    return model

# recover any VRAM left by a previous (failed) run in the same kernel
_free = free_gpu("bin_model", "gp_model", "FEAT", "model", "full")
if _free is not None and _free < 10:
    print("\n" + "!" * 64)
    print("!! Less than 10 GB free BEFORE training -> a previous run left memory")
    print("!! stranded on the GPU. pip/empty_cache cannot recover it.")
    print("!! FIX: top menu -> Run -> 'Restart & run all'  (or 'Factory reset'),")
    print("!!      then run from Stage 1. Do NOT just re-run this cell.")
    print("!" * 64 + "\n")

bin_model = load_dinov2(2)
bin_model = train_classifier(bin_model, TRAIN_CSV, VAL_CSV, 2, "binary_label",
                             WORK / "binary_classifier_best.pt", monitor="auroc")
torch.save(bin_model.state_dict(), WORK / "binary_classifier_final.pt")
print("Stage 6 complete.")

GPU free: 15.5 / 15.6 GB


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

frozen ep0:   0%|          | 0/1528 [00:00<?, ?it/s]

[frozen] ep0 train_loss=0.2927 val_loss=0.5839 val_acc=0.7463 val_auroc=0.9193 val_qwk=0.4883


frozen ep1:   0%|          | 0/1528 [00:00<?, ?it/s]

[frozen] ep1 train_loss=0.2663 val_loss=1.0081 val_acc=0.6616 val_auroc=0.9254 val_qwk=0.3144


... (14 outputs left out)

frozen ep9:   0%|          | 0/1528 [00:00<?, ?it/s]

[frozen] ep9 train_loss=0.2555 val_loss=0.3505 val_acc=0.8722 val_auroc=0.9434 val_qwk=0.7444
  fine-tuning last 2 blocks: 14.2M trainable params


finetune ep0:   0%|          | 0/1528 [00:00<?, ?it/s]

[finetune] ep0 train_loss=0.2081 val_loss=0.9151 val_acc=0.7102 val_auroc=0.9473 val_qwk=0.4138
  early stop (finetune)
Stage 6 complete.


## Stage 7 — DINOv2 GP (3-class) classifier, transfer from binary backbone

In [9]:
def make_gp_csv(src_csv, out_csv, cap=None):
    """Keep malignant patches, remap GP3/4/5 -> 0/1/2 in a gp_label column."""
    df = pd.read_csv(src_csv)
    df = df[df["binary_label"] == 1].copy()
    df["gp_label"] = df["label"].map({"GP3": 0, "GP4": 1, "GP5": 2})
    df = df.dropna(subset=["gp_label"]); df["gp_label"] = df["gp_label"].astype(int)
    if cap:
        df = df.head(cap)
    df.to_csv(out_csv, index=False)
    return out_csv

GP_TRAIN = make_gp_csv(TRAIN_CSV, WORK / "sicap_gp_train.csv")
GP_VAL   = make_gp_csv(VAL_CSV,   WORK / "sicap_gp_val.csv")
print("GP train/val:", len(pd.read_csv(GP_TRAIN)), len(pd.read_csv(GP_VAL)))

free_gpu("model")  # reclaim memory before building the GP model
gp_model = load_dinov2(3)
# transfer backbone weights from the binary checkpoint (skip head.*)
bin_sd = torch.load(WORK / "binary_classifier_best.pt", map_location=DEVICE)
gp_model.load_state_dict({k: v for k, v in bin_sd.items() if not k.startswith("head.")}, strict=False)

gp_model = train_classifier(gp_model, GP_TRAIN, GP_VAL, 3, "gp_label",
                            WORK / "gp_classifier_best.pt", monitor="qwk")
print("Stage 7 complete.")

GP train/val: 14414 1101
GPU free: 14.9 / 15.6 GB


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

frozen ep0:   0%|          | 0/901 [00:00<?, ?it/s]

[frozen] ep0 train_loss=0.5364 val_loss=0.5964 val_acc=0.7620 val_auroc=0.8856 val_qwk=0.6533


frozen ep1:   0%|          | 0/901 [00:00<?, ?it/s]

[frozen] ep1 train_loss=0.4429 val_loss=0.5269 val_acc=0.7975 val_auroc=0.8989 val_qwk=0.6747


... (6 outputs left out)

frozen ep5:   0%|          | 0/901 [00:00<?, ?it/s]

[frozen] ep5 train_loss=0.4495 val_loss=0.6808 val_acc=0.7357 val_auroc=0.8827 val_qwk=0.6235
  early stop (frozen)
  fine-tuning last 2 blocks: 14.2M trainable params


finetune ep0:   0%|          | 0/901 [00:00<?, ?it/s]

[finetune] ep0 train_loss=0.3454 val_loss=0.5767 val_acc=0.7838 val_auroc=0.8927 val_qwk=0.6662
  early stop (finetune)
Stage 7 complete.


## Stage 8 — DINOv2 embedding extraction (SICAP slides + PANDA WSIs)

In [10]:
def load_backbone(ckpt):
    """Load a trained classifier and return just the DINOv2 backbone (eval)."""
    m = load_dinov2(2)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    return m

free_gpu("bin_model", "gp_model", "model")  # only the feature extractor is needed now
FEAT = load_backbone(WORK / "binary_classifier_best.pt")
_tf = eval_tf()

@torch.no_grad()
def extract_embeddings(paths):
    """Return (N,768) CLS embeddings for a list of image paths."""
    embs = []
    for i in range(0, len(paths), BATCH_SIZE_PATCH):
        batch = []
        for p in paths[i:i + BATCH_SIZE_PATCH]:
            batch.append(_tf(Image.open(p).convert("RGB")))
        x = torch.stack(batch).to(DEVICE)
        with torch.cuda.amp.autocast():
            f = FEAT.backbone(x)
        embs.append(f.float().cpu().numpy())
    return np.concatenate(embs, 0) if embs else np.zeros((0, EMBED_DIM), np.float32)

def extract_sicap_split(csv_path, split):
    """Group patches by slide; save 10x/20x embeddings + coords per slide."""
    out = WORK / f"embeddings/sicap/{split}"; out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(csv_path)
    df["slide_id"] = df["filename"].apply(get_slide_id)
    for sid, g in tqdm(df.groupby("slide_id"), desc=f"emb sicap/{split}"):
        g = g.reset_index(drop=True)
        np.save(out / f"{sid}_10x.npy", extract_embeddings(g["patch_path_10x"].tolist()))
        np.save(out / f"{sid}_20x.npy", extract_embeddings(g["patch_path_20x"].tolist()))
        np.save(out / f"{sid}_coords.npy",
                np.array([get_coords(f) for f in g["filename"]], dtype=np.float32))
    return out

extract_sicap_split(TRAIN_CSV, "train")
extract_sicap_split(VAL_CSV,   "val")

# ---- PANDA: tile WSIs on the fly ----
def open_slide(path):
    try:
        import openslide
        return openslide.OpenSlide(str(path))
    except Exception as e:
        print("openslide fail", path.name, e); return None

@torch.no_grad()
def extract_panda(cap=None):
    """Tile PANDA WSIs at level 0, normalise, build 10x/20x, save embeddings."""
    out = WORK / "embeddings/panda"; out.mkdir(parents=True, exist_ok=True)
    if not PANDA_CSV.exists():
        print("PANDA csv missing; skipping Stage 8 PANDA."); return out
    ids = pd.read_csv(PANDA_CSV)["image_id"].tolist()
    if cap:
        ids = ids[:cap]
    for n, sid in enumerate(tqdm(ids, desc="emb panda")):
        wsi = PANDA_IMG / f"{sid}.tiff"
        if not wsi.exists():
            continue
        if (out / f"{sid}_10x.npy").exists():
            continue
        slide = open_slide(wsi)
        if slide is None:
            continue
        try:
            W, H = slide.level_dimensions[0]
            e10, e20, coords = [], [], []
            stop = False
            for yy in range(0, H - PATCH_SIZE, PATCH_SIZE):
                if stop:
                    break
                for xx in range(0, W - PATCH_SIZE, PATCH_SIZE):
                    tile = np.array(slide.read_region((xx, yy), 0, (PATCH_SIZE, PATCH_SIZE)).convert("RGB"))
                    if tissue_fraction(cv2.cvtColor(tile, cv2.COLOR_RGB2BGR)) < TISSUE_THRESHOLD:
                        continue
                    tile = normalise_patch(tile)
                    im = Image.fromarray(tile)
                    t10 = im.resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)
                    c = (PATCH_SIZE - 256) // 2
                    t20 = im.crop((c, c, c + 256, c + 256)).resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)
                    e10.append(_tf(t10)); e20.append(_tf(t20))
                    coords.append((yy // PATCH_SIZE, xx // PATCH_SIZE))
                    if len(e10) >= MAX_TILES_PER_SLIDE:   # cap tiles per WSI for speed
                        stop = True; break
            if not e10:
                slide.close(); continue
            def _embed(tensors):
                outs = []
                for i in range(0, len(tensors), BATCH_SIZE_PATCH):
                    x = torch.stack(tensors[i:i + BATCH_SIZE_PATCH]).to(DEVICE)
                    with torch.cuda.amp.autocast():
                        outs.append(FEAT.backbone(x).float().cpu().numpy())
                return np.concatenate(outs, 0)
            np.save(out / f"{sid}_10x.npy", _embed(e10))
            np.save(out / f"{sid}_20x.npy", _embed(e20))
            np.save(out / f"{sid}_coords.npy", np.array(coords, dtype=np.float32))
            slide.close()
        except Exception as e:
            print("panda tile error", sid, e); traceback.print_exc()
        if (n + 1) % 50 == 0:
            print(f"  processed {n + 1} slides")
    return out

extract_panda(MAX_PANDA_SLIDES)
print("Stage 8 complete.")

GPU free: 15.1 / 15.6 GB


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

emb sicap/train:   0%|          | 0/366 [00:00<?, ?it/s]

emb sicap/val:   0%|          | 0/33 [00:00<?, ?it/s]

emb panda:   0%|          | 0/300 [00:00<?, ?it/s]

  processed 50 slides
  processed 100 slides
  processed 150 slides
  processed 200 slides
  processed 250 slides
  processed 300 slides
Stage 8 complete.


## Stage 9 — Slide-level ISUP labels (SICAP >5% rule; PANDA from csv)

In [11]:
ISUP_MAP = {("GP3", "GP3"): 1, ("GP3", "GP4"): 2, ("GP4", "GP3"): 2,
            ("GP4", "GP4"): 3, ("GP4", "GP5"): 4, ("GP5", "GP4"): 4,
            ("GP3", "GP5"): 3, ("GP5", "GP3"): 3, ("GP5", "GP5"): 5}

def slide_labels(csv_path, out_csv):
    """Per-slide ISUP grade. Prefers wsi_labels.xlsx ground truth (SID2ISUP);
    falls back to the >5% primary/secondary GP rule for unmatched slides."""
    gt = SID2ISUP if "SID2ISUP" in globals() else {}
    df = pd.read_csv(csv_path)
    df["slide_id"] = df["filename"].apply(get_slide_id)
    rows, n_gt = [], 0
    for sid, g in df.groupby("slide_id"):
        mal = g[g["binary_label"] == 1]
        nm = len(mal)
        # mask-derived primary/secondary (kept for reference + fallback)
        if nm == 0:
            primary, secondary, isup_rule = "None", "None", 0
        else:
            pct = (mal["label"].value_counts() / nm * 100)
            gps = list(pct[pct > 5].sort_values(ascending=False).index)
            primary = gps[0] if gps else mal["label"].value_counts().index[0]
            secondary = gps[1] if len(gps) > 1 else primary
            isup_rule = ISUP_MAP.get((primary, secondary), ISUP_MAP.get((primary, primary), 1))
        if sid in gt:                          # ground-truth ISUP wins
            isup = int(gt[sid]); n_gt += 1
        else:
            isup = int(isup_rule)
        rows.append({"slide_id": sid, "primary_gp": primary, "secondary_gp": secondary,
                     "isup_grade": isup, "num_patches": len(g), "num_malignant_patches": nm})
    out = pd.DataFrame(rows); out.to_csv(out_csv, index=False)
    print(f"  [{Path(out_csv).name}] {len(out)} slides | {n_gt} from ground truth, "
          f"{len(out) - n_gt} from >5% rule")
    return out

s_tr = slide_labels(TRAIN_CSV, WORK / "sicap_slide_labels_train.csv")
s_va = slide_labels(VAL_CSV,   WORK / "sicap_slide_labels_val.csv")

# PANDA slide labels from provided csv
if PANDA_CSV.exists():
    pcsv = pd.read_csv(PANDA_CSV).rename(columns={"image_id": "slide_id"})[["slide_id", "isup_grade"]]
    # restrict to slides we actually embedded
    emb_ids = {p.name.replace("_10x.npy", "") for p in (WORK / "embeddings/panda").glob("*_10x.npy")}
    if emb_ids:
        pcsv = pcsv[pcsv["slide_id"].isin(emb_ids)]
    pcsv.to_csv(WORK / "panda_slide_labels.csv", index=False)
    print("PANDA ISUP dist:\n", pcsv["isup_grade"].value_counts().sort_index())

print("SICAP train ISUP:\n", s_tr["isup_grade"].value_counts().sort_index())
print("SICAP val ISUP:\n", s_va["isup_grade"].value_counts().sort_index())
print("Stage 9 complete.")

  [sicap_slide_labels_train.csv] 366 slides | 122 from ground truth, 244 from >5% rule
  [sicap_slide_labels_val.csv] 33 slides | 33 from ground truth, 0 from >5% rule
PANDA ISUP dist:
 isup_grade
0    75
1    80
2    38
3    39
4    35
5    33
Name: count, dtype: int64
SICAP train ISUP:
 isup_grade
0     51
1     68
2     79
3    115
4     21
5     32
Name: count, dtype: int64
SICAP val ISUP:
 isup_grade
0     9
1     5
2     3
3     3
4     1
5    12
Name: count, dtype: int64
Stage 9 complete.


## Stage 10 — Graph construction (multi-scale nodes, KNN + cross-scale edges)

In [12]:
from torch_geometric.data import Data
try:
    import torch_cluster  # noqa: F401  (only present if companion wheel installed)
    from torch_geometric.nn import knn_graph
    _HAVE_KNN = True
except Exception:
    _HAVE_KNN = False      # -> numpy KNN fallback (no torch_cluster needed)
print("KNN backend:", "torch_cluster" if _HAVE_KNN else "numpy fallback")

def _knn_edges(coords, k):
    """KNN edge_index from (N,2) coords; numpy fallback if torch_cluster absent."""
    N = coords.shape[0]
    k = min(k, max(1, N - 1))
    if N <= 1:
        return torch.zeros((2, 0), dtype=torch.long)
    if _HAVE_KNN:
        return knn_graph(torch.tensor(coords, dtype=torch.float), k=k, loop=False)
    d = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    np.fill_diagonal(d, np.inf)
    nn_idx = np.argsort(d, axis=1)[:, :k]
    src = np.repeat(np.arange(N), k); dst = nn_idx.reshape(-1)
    return torch.tensor(np.vstack([src, dst]), dtype=torch.long)

def build_slide_graph(slide_id, split, isup_grade):
    """Return a PyG Data with 2N multi-scale nodes and within/cross-scale edges."""
    base = WORK / f"embeddings/{'sicap/' + split if split in ('train', 'val') else 'panda'}"
    e10 = np.load(base / f"{slide_id}_10x.npy")
    e20 = np.load(base / f"{slide_id}_20x.npy")
    coords = np.load(base / f"{slide_id}_coords.npy")
    N = min(len(e10), len(e20), len(coords))
    if N == 0:
        return None
    e10, e20, coords = e10[:N], e20[:N], coords[:N]
    cn = coords - coords.mean(0, keepdims=True)
    cn = cn / (np.abs(cn).max() + 1e-6)
    x10 = np.concatenate([e10, np.zeros((N, 1)), cn], 1)   # scale flag 0
    x20 = np.concatenate([e20, np.ones((N, 1)), cn], 1)    # scale flag 1
    x = torch.tensor(np.concatenate([x10, x20], 0), dtype=torch.float)  # (2N, 771)
    e_intra10 = _knn_edges(coords, KNN_K)
    e_intra20 = _knn_edges(coords, KNN_K) + N
    cross = torch.stack([torch.arange(N), torch.arange(N) + N])
    cross = torch.cat([cross, cross.flip(0)], 1)
    edge_index = torch.cat([e_intra10, e_intra20, cross], 1)
    return Data(x=x, edge_index=edge_index,
                y=torch.tensor([int(isup_grade)], dtype=torch.long), slide_id=slide_id)

def build_all_graphs(slide_csv, split, cap=None):
    out = WORK / f"graphs/{split if split in ('train','val') else 'panda'}"
    if split == "panda":
        out = WORK / "graphs/panda"
    else:
        out = WORK / f"graphs/sicap/{split}"
    out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(slide_csv)
    if cap:
        df = df.head(cap)
    n, nodes = 0, []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"graphs {split}"):
        try:
            g = build_slide_graph(str(r["slide_id"]), split, int(r["isup_grade"]))
        except FileNotFoundError:
            continue
        if g is None:
            continue
        torch.save(g, out / f"{r['slide_id']}.pt")
        nodes.append(g.x.shape[0]); n += 1
    print(f"{split}: {n} graphs, avg nodes={np.mean(nodes):.1f}" if nodes else f"{split}: 0 graphs")
    return out

build_all_graphs(WORK / "sicap_slide_labels_train.csv", "train", MAX_GRAPH_SLIDES)
build_all_graphs(WORK / "sicap_slide_labels_val.csv",   "val",   MAX_GRAPH_SLIDES)
if (WORK / "panda_slide_labels.csv").exists():
    build_all_graphs(WORK / "panda_slide_labels.csv", "panda", MAX_PANDA_SLIDES)
print("Stage 10 complete.")

KNN backend: numpy fallback


graphs train:   0%|          | 0/366 [00:00<?, ?it/s]

train: 366 graphs, avg nodes=133.6


graphs val:   0%|          | 0/33 [00:00<?, ?it/s]

val: 33 graphs, avg nodes=130.9


graphs panda:   0%|          | 0/300 [00:00<?, ?it/s]

panda: 300 graphs, avg nodes=398.5
Stage 10 complete.


## Stage 11 — PyG dataset / loaders with on-the-fly graph augmentation

In [13]:
from torch_geometric.loader import DataLoader as GeoLoader

def augment_graph(data):
    """Node dropout 10%, coord jitter std=0.01, edge dropout 10%."""
    data = data.clone()
    N = data.x.shape[0]
    keep = torch.rand(N) > 0.10
    if keep.sum() < 2:
        keep = torch.ones(N, dtype=torch.bool)
    idx_map = -torch.ones(N, dtype=torch.long)
    idx_map[keep] = torch.arange(keep.sum())
    data.x = data.x[keep].clone()
    data.x[:, -2:] += torch.randn_like(data.x[:, -2:]) * 0.01
    ei = data.edge_index
    mask = keep[ei[0]] & keep[ei[1]]
    ei = ei[:, mask]
    ei = torch.stack([idx_map[ei[0]], idx_map[ei[1]]])
    if ei.shape[1] > 0:
        emask = torch.rand(ei.shape[1]) > 0.10
        ei = ei[:, emask]
    data.edge_index = ei
    return data

class SlideGraphDataset(torch.utils.data.Dataset):
    """Loads pre-built .pt graphs from a directory; optional augmentation."""
    def __init__(self, root, augment=False, ids=None):
        self.files = sorted(Path(root).glob("*.pt"))
        if ids is not None:
            ids = set(ids); self.files = [f for f in self.files if f.stem in ids]
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        data = torch.load(self.files[i], weights_only=False)
        return augment_graph(data) if self.augment else data

def graph_loader(root, augment, shuffle, ids=None, bs=BATCH_SIZE_GRAPH):
    ds = SlideGraphDataset(root, augment, ids)
    return GeoLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS)

train_gl = graph_loader(WORK / "graphs/sicap/train", True, True)
val_gl   = graph_loader(WORK / "graphs/sicap/val",   False, False)
panda_gl = graph_loader(WORK / "graphs/panda",       False, False) if (WORK / "graphs/panda").exists() else None
print("Sizes -> train:", len(train_gl.dataset), "val:", len(val_gl.dataset),
      "panda:", len(panda_gl.dataset) if panda_gl else 0)
if len(train_gl.dataset):
    b = next(iter(train_gl))
    print("Batch: graphs=", b.num_graphs, "nodes=", b.x.shape[0], "labels=", b.y.tolist())
print("Stage 11 complete.")

Sizes -> train: 366 val: 33 panda: 300
Batch: graphs= 4 nodes= 482 labels= [3, 3, 3, 0]
Stage 11 complete.


## Stage 12 — Graph Transformer + ILRA-MIL model

In [14]:
from torch_geometric.nn import TransformerConv, global_mean_pool
from torch_geometric.utils import to_dense_batch

class GraphTransformerLayer(nn.Module):
    """Pre-norm TransformerConv block + FFN with residuals."""
    def __init__(self, hidden_dim, num_heads, dropout=DROPOUT):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv = TransformerConv(hidden_dim, hidden_dim // num_heads,
                                    heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(hidden_dim * 4, hidden_dim))

    def forward(self, x, edge_index):
        x = x + self.conv(self.norm1(x), edge_index)
        x = x + self.ffn(self.norm2(x))
        return x

class ILRAPooling(nn.Module):
    """Cross-attention pooling: learnable latents attend over slide nodes."""
    def __init__(self, hidden_dim, num_latent=ILRA_LATENT, num_heads=4, dropout=DROPOUT):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_latent, hidden_dim) * 0.02)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.last_attn = None

    def forward(self, x, batch):
        dense, mask = to_dense_batch(x, batch)          # (B, Nmax, H), (B, Nmax)
        B = dense.shape[0]
        q = self.latents.unsqueeze(0).expand(B, -1, -1)  # (B, L, H)
        out, attn = self.attn(q, dense, dense, key_padding_mask=~mask, need_weights=True)
        self.last_attn = attn.detach()                  # (B, L, Nmax)
        return self.norm(out).mean(1)                   # (B, H)

class GraphTransformerILRA(nn.Module):
    """Input proj -> N transformer layers -> ILRA pooling -> classifier MLP."""
    def __init__(self, node_feat_dim=NODE_FEAT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 num_heads=NUM_HEADS, ilra_latent=ILRA_LATENT, dropout=DROPOUT,
                 num_classes=6, pool="ilra"):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(node_feat_dim, hidden_dim),
                                   nn.LayerNorm(hidden_dim), nn.GELU())
        self.layers = nn.ModuleList([GraphTransformerLayer(hidden_dim, num_heads, dropout)
                                     for _ in range(num_layers)])
        self.pool_type = pool
        self.ilra = ILRAPooling(hidden_dim, ilra_latent, 4, dropout) if pool != "mean" else None
        self.cls = nn.Sequential(nn.Linear(hidden_dim, 256), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, data):
        x = self.proj(data.x)
        for l in self.layers:
            x = l(x, data.edge_index)
        if self.pool_type == "mean":
            emb = global_mean_pool(x, data.batch)
        else:
            emb = self.ilra(x, data.batch)
        return self.cls(emb)

# ---- graph objective helpers (ordinal regression directly optimizes QWK) ----
NUM_ISUP = 6
def graph_out_dim():
    """Output dim of the graph head: 1 for ordinal regression, 6 for cross-entropy."""
    return 1 if GRAPH_OBJECTIVE == "ordinal" else NUM_ISUP

def graph_loss(out, y):
    """MSE on the continuous ISUP score (ordinal) or cross-entropy (ce)."""
    if GRAPH_OBJECTIVE == "ordinal":
        return F.mse_loss(out.squeeze(-1), y.float())
    return F.cross_entropy(out, y)

def graph_pred(out):
    """Predicted ISUP grade: round+clamp the score (ordinal) or argmax (ce)."""
    if GRAPH_OBJECTIVE == "ordinal":
        return out.squeeze(-1).round().clamp(0, NUM_ISUP - 1).long()
    return out.argmax(1)

def graph_probs(out):
    """Per-grade probabilities. Softmax (ce); distance-based pseudo-probs (ordinal)."""
    if GRAPH_OBJECTIVE != "ordinal":
        return torch.softmax(out.float(), 1)
    s = out.squeeze(-1).float().clamp(0, NUM_ISUP - 1)
    grades = torch.arange(NUM_ISUP, device=out.device).float()
    return torch.softmax(-(s[:, None] - grades[None, :]).abs(), 1)

_m = GraphTransformerILRA(num_classes=graph_out_dim()).to(DEVICE)
print(f"Params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M | objective={GRAPH_OBJECTIVE} "
      f"(out_dim={graph_out_dim()})")
print("Stage 12 complete.")

Params: 11.10M | objective=ordinal (out_dim=1)
Stage 12 complete.


## Stage 13 — Graph Transformer training with patient-level 5-fold CV

In [15]:
from sklearn.model_selection import StratifiedKFold

def patient_id(slide_id):
    """Patient = slide id up to the last underscore (falls back to whole id)."""
    s = str(slide_id)                      # slide_id may be read back from CSV as int
    return s.rsplit("_", 1)[0] if "_" in s else s

def train_graph_model(train_root, ids_train, ids_val, num_classes=6, pool="ilra",
                      ckpt=None, epochs=None):
    """Train one GraphTransformerILRA on the given slide-id subsets; early stop on QWK."""
    epochs = epochs or EPOCHS_GRAPH
    tl = graph_loader(train_root, True, True, ids_train)
    vl = graph_loader(train_root, False, False, ids_val)
    if len(tl.dataset) == 0 or len(vl.dataset) == 0:
        print("  empty fold, skipping"); return None, -1
    model = GraphTransformerILRA(num_classes=graph_out_dim(), pool=pool).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    warm = torch.optim.lr_scheduler.LinearLR(opt, 0.1, 1.0, total_iters=2)
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs - 2), eta_min=1e-6)
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[2])
    scaler = torch.cuda.amp.GradScaler()
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train(); tl_losses = []
        for b in tl:
            b = b.to(DEVICE); opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = graph_loss(model(b), b.y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl_losses.append(loss.item())
        sched.step()
        model.eval(); ys, ps, vlosses = [], [], []
        with torch.no_grad():
            for b in vl:
                b = b.to(DEVICE)
                with torch.cuda.amp.autocast():
                    out = model(b); vlosses.append(graph_loss(out, b.y).item())
                ys += b.y.cpu().tolist(); ps += graph_pred(out).cpu().tolist()
        qwk = cohen_kappa_score(ys, ps, weights="quadratic", labels=list(range(num_classes))) if len(set(ys)) > 1 else 0.0
        print(f"  ep{ep} train_loss={np.mean(tl_losses):.4f} val_loss={np.mean(vlosses):.4f} val_qwk={qwk:.4f}")
        if qwk > best:
            best, best_state, bad = qwk, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
            if ckpt:
                torch.save(best_state, ckpt)
        else:
            bad += 1
            if bad >= PATIENCE_GRAPH:
                print("  early stop"); break
    if best_state:
        model.load_state_dict(best_state)
    return model, best

slides = pd.read_csv(WORK / "sicap_slide_labels_train.csv")
slides["slide_id"] = slides["slide_id"].astype(str)   # keep ids as strings everywhere
_sid2pat = SID2PATIENT if "SID2PATIENT" in globals() else {}
slides["patient"] = slides["slide_id"].map(lambda s: _sid2pat.get(str(s), patient_id(s)))
# stratify patients by their majority ISUP grade
pat = slides.groupby("patient")["isup_grade"].agg(lambda s: s.value_counts().index[0]).reset_index()
fold_qwks, best_overall, best_overall_qwk, best_fold = [], None, -1, -1
from sklearn.model_selection import KFold
_pats = pat["patient"].values
_grades = pat["isup_grade"].values
_min_class = int(pd.Series(_grades).value_counts().min())
_n_uniq = len(set(_grades))
if len(pat) < 2:
    # degenerate (smoke: 1 patient) -> train==val so the stage still runs
    splitter = [(np.arange(len(pat)), np.arange(len(pat)))]
elif _n_uniq > 1 and _min_class >= 2:
    # stratify only when every grade has >= folds members
    k = max(2, min(N_FOLDS, _min_class, len(pat)))
    splitter = list(StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED).split(_pats, _grades))
    print(f"  StratifiedKFold: {k} folds ({len(pat)} patients, min class={_min_class})")
else:
    # too few members per grade to stratify -> plain patient-level KFold
    k = max(2, min(N_FOLDS, len(pat)))
    splitter = list(KFold(n_splits=k, shuffle=True, random_state=SEED).split(_pats))
    print(f"  KFold (non-stratified): {k} folds ({len(pat)} patients, min class={_min_class})")

for fold, (tr_idx, va_idx) in enumerate(splitter):
    tr_pat = set(pat["patient"].iloc[tr_idx]); va_pat = set(pat["patient"].iloc[va_idx])
    ids_tr = slides[slides["patient"].isin(tr_pat)]["slide_id"].astype(str).tolist()
    ids_va = slides[slides["patient"].isin(va_pat)]["slide_id"].astype(str).tolist()
    print(f"Fold {fold}: train slides={len(ids_tr)} val slides={len(ids_va)}")
    m, q = train_graph_model(WORK / "graphs/sicap/train", ids_tr, ids_va,
                             ckpt=WORK / f"graph_model_fold{fold}.pt")
    fold_qwks.append(q)
    if q > best_overall_qwk:
        best_overall_qwk, best_fold = q, fold

if fold_qwks:
    print(f"\n5-fold val QWK: mean={np.mean(fold_qwks):.4f} std={np.std(fold_qwks):.4f}")
FINAL_GRAPH_CKPT = WORK / f"graph_model_fold{max(best_fold,0)}.pt"
print("Best fold:", best_fold, "->", FINAL_GRAPH_CKPT)
print("Stage 13 complete.")

  StratifiedKFold: 5 folds (320 patients, min class=16)
Fold 0: train slides=295 val slides=71
  ep0 train_loss=2.5114 val_loss=1.3876 val_qwk=0.2746
  ep1 train_loss=1.2388 val_loss=1.0264 val_qwk=0.3979
  ep2 train_loss=0.8192 val_loss=0.7085 val_qwk=0.6984
  ep3 train_loss=0.6910 val_loss=0.5896 val_qwk=0.8071
  ep4 train_loss=0.5932 val_loss=0.6103 val_qwk=0.7734
  ep5 train_loss=0.4283 val_loss=0.4724 val_qwk=0.8010
  ep6 train_loss=0.3466 val_loss=0.5386 val_qwk=0.8153
  ep7 train_loss=0.3195 val_loss=0.5813 val_qwk=0.7902
  ep8 train_loss=0.3023 val_loss=0.6807 val_qwk=0.8008
  ep9 train_loss=0.2858 val_loss=0.5057 val_qwk=0.8240
  ep10 train_loss=0.2668 val_loss=0.6233 val_qwk=0.8122
  ep11 train_loss=0.6818 val_loss=0.5606 val_qwk=0.8304
  ep12 train_loss=0.2625 val_loss=0.4721 val_qwk=0.8020
  ep13 train_loss=0.1974 val_loss=0.4872 val_qwk=0.8678
  ep14 train_loss=0.1554 val_loss=0.4712 val_qwk=0.8717
  ep15 train_loss=0.1506 val_loss=0.4810 val_qwk=0.8258
  ep16 train_loss=0

## Stage 14 — Ablation study (5 variants, bootstrapped QWK with 95% CI)

In [16]:
def bootstrap_qwk(y_true, y_pred, n=N_BOOTSTRAP):
    """QWK plus a percentile 95% CI from bootstrap resampling of slides."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    if len(y_true) == 0:
        return 0.0, (0.0, 0.0)
    base = cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=list(range(6)))
    rng = np.random.RandomState(SEED); stats = []
    for _ in range(n):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(set(y_true[idx])) < 2:
            continue
        stats.append(cohen_kappa_score(y_true[idx], y_pred[idx], weights="quadratic", labels=list(range(6))))
    if not stats:
        return base, (base, base)
    return base, (float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5)))

@torch.no_grad()
def graph_predict(model, loader):
    """Return (true, pred) ISUP arrays for a graph loader."""
    model.eval(); ys, ps = [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = model(b)
        ys += b.y.cpu().tolist(); ps += graph_pred(out).cpu().tolist()
    return ys, ps

def load_fold_ensemble(num_outputs=None):
    """Load all saved fold checkpoints as an ensemble (averaged at inference)."""
    num_outputs = num_outputs or graph_out_dim()
    models = []
    for cp in sorted(WORK.glob("graph_model_fold*.pt")):
        m = GraphTransformerILRA(num_classes=num_outputs).to(DEVICE)
        try:
            m.load_state_dict(torch.load(cp, map_location=DEVICE)); m.eval()
            models.append(m)
        except Exception as e:
            print("  skip ckpt", cp.name, e)
    return models

@torch.no_grad()
def ensemble_predict(models, loader):
    """Average fold outputs, then derive (true, pred, probs) per slide."""
    for m in models:
        m.eval()
    ys, ps, prob = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = torch.stack([m(b).float() for m in models]).mean(0)
        ys += b.y.cpu().tolist(); ps += graph_pred(out).cpu().tolist()
        prob.append(graph_probs(out).cpu().numpy())
    return ys, ps, (np.concatenate(prob, 0) if prob else np.zeros((0, NUM_ISUP)))

@torch.no_grad()
def ensemble_scores(models, loader):
    """Return (true, continuous_score) from the averaged ordinal regression head."""
    for m in models:
        m.eval()
    ys, sc = [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = torch.stack([m(b).float() for m in models]).mean(0)
        ys += b.y.cpu().tolist(); sc += out.squeeze(-1).cpu().tolist()
    return np.array(ys), np.array(sc)

class OptimizedRounder:
    """Fit ISUP cut-points on validation scores to maximise QWK (PANDA-style).
    Lifts exact accuracy of the ordinal head without retraining (caveat #3)."""
    def __init__(self, n=NUM_ISUP):
        self.coef = np.arange(n - 1) + 0.5
    def _qwk(self, coef, X, y):
        return -cohen_kappa_score(y, np.digitize(X, np.sort(coef)),
                                  weights="quadratic", labels=list(range(NUM_ISUP)))
    def fit(self, X, y):
        import scipy.optimize as spo
        try:
            r = spo.minimize(self._qwk, self.coef, args=(X, y), method="Nelder-Mead")
            self.coef = np.sort(r.x)
        except Exception as e:
            print("  threshold calibration failed, using defaults:", e)
        return self
    def predict(self, X):
        return np.digitize(X, self.coef)

# --- Variant 1/2: base-paper >5% rule, applied to saved embeddings ---
# Uses the trained patch-classifier HEADS on the cached Phikon embeddings, so the
# rule can be scored on BOTH SICAP val AND PANDA (apples-to-apples vs the graph).
@torch.no_grad()
def aggregate_from_embeddings(split, multiscale=False):
    """Base-paper >5% aggregation from cached embeddings. split: 'sicap_val'|'panda'."""
    bm = load_dinov2(2); bm.load_state_dict(torch.load(WORK / "binary_classifier_best.pt", map_location=DEVICE)); bm.eval()
    gp = load_dinov2(3); gp.load_state_dict(torch.load(WORK / "gp_classifier_best.pt", map_location=DEVICE)); gp.eval()
    if split == "panda":
        base = WORK / "embeddings/panda"
        gt = pd.read_csv(WORK / "panda_slide_labels.csv").set_index("slide_id")["isup_grade"].to_dict()
    else:
        base = WORK / "embeddings/sicap/val"
        gt = pd.read_csv(WORK / "sicap_slide_labels_val.csv").set_index("slide_id")["isup_grade"].to_dict()
    yt, yp = [], []
    for sid, isup in gt.items():
        f10 = base / f"{sid}_10x.npy"
        if not f10.exists():
            continue
        e10 = np.load(f10)
        emb = e10
        if multiscale and (base / f"{sid}_20x.npy").exists():
            e20 = np.load(base / f"{sid}_20x.npy")
            n = min(len(e10), len(e20)); emb = (e10[:n] + e20[:n]) / 2.0
        t = torch.tensor(emb, dtype=torch.float, device=DEVICE)
        with torch.cuda.amp.autocast():
            mal = bm.head(t).argmax(1).cpu().numpy()
            gprob = torch.softmax(gp.head(t).float(), 1).cpu().numpy()
        gps = [["GP3", "GP4", "GP5"][int(gprob[i].argmax())] for i in range(len(mal)) if mal[i] == 1]
        if not gps:
            yt.append(int(isup)); yp.append(0); continue
        pct = pd.Series(gps).value_counts(); pct = pct / pct.sum() * 100
        gl = list(pct[pct > 5].sort_values(ascending=False).index)
        pr = gl[0] if gl else pd.Series(gps).value_counts().index[0]
        se = gl[1] if len(gl) > 1 else pr
        yt.append(int(isup)); yp.append(ISUP_MAP.get((pr, se), ISUP_MAP.get((pr, pr), 1)))
    return yt, yp

def _variant_sicap_panda(name, multiscale):
    sv = bootstrap_qwk(*aggregate_from_embeddings("sicap_val", multiscale))
    pv = (None, (None, None))
    if (WORK / "embeddings/panda").exists():
        pv = bootstrap_qwk(*aggregate_from_embeddings("panda", multiscale))
    return (name, sv[0], sv[1], pv[0], pv[1])

results = []
try:
    results.append(_variant_sicap_panda("V1 Base Replicated", False))
    results.append(_variant_sicap_panda("V2 +Multi-Scale", True))
except Exception as e:
    print("patch-aggregation variant skipped:", e); traceback.print_exc()

# --- Variant 3: graph + mean pool, Variant 4: ILRA-only(no graph layers), Variant 5: full ---
tr_ids = pd.read_csv(WORK / "sicap_slide_labels_train.csv")["slide_id"].astype(str).tolist()
va_ids = pd.read_csv(WORK / "sicap_slide_labels_val.csv")["slide_id"].astype(str).tolist()

def train_eval_variant(name, pool, num_layers):
    """Train a graph variant on full SICAP train, eval on SICAP val + PANDA."""
    global GraphTransformerILRA
    orig = GraphTransformerILRA.__init__
    m = GraphTransformerILRA(num_classes=graph_out_dim(), pool=pool, num_layers=num_layers).to(DEVICE)
    tl = graph_loader(WORK / "graphs/sicap/train", True, True, tr_ids)
    vl = graph_loader(WORK / "graphs/sicap/val", False, False, va_ids)
    opt = torch.optim.AdamW(m.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler()
    for ep in range(EPOCHS_GRAPH):
        m.train()
        for b in tl:
            b = b.to(DEVICE); opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = graph_loss(m(b), b.y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    yt, yp = graph_predict(m, vl)
    sv = bootstrap_qwk(yt, yp)
    pv = (None, (None, None))
    if panda_gl and len(panda_gl.dataset):
        pyt, pyp = graph_predict(m, panda_gl); pv = bootstrap_qwk(pyt, pyp)
    return (name, sv[0], sv[1], pv[0], pv[1])

try:
    results.append(train_eval_variant("V3 Graph+MeanPool", "mean", NUM_LAYERS))
    results.append(train_eval_variant("V4 ILRA-only(no graph)", "ilra", 0))
except Exception as e:
    print("graph variants skipped:", e)

# Variant 5a = full model, SINGLE best fold (isolates the graph from the ensemble)
if FINAL_GRAPH_CKPT.exists():
    _single = GraphTransformerILRA(num_classes=graph_out_dim()).to(DEVICE)
    _single.load_state_dict(torch.load(FINAL_GRAPH_CKPT, map_location=DEVICE)); _single.eval()
    yt, yp = graph_predict(_single, val_gl); sv = bootstrap_qwk(yt, yp)
    pv = (None, (None, None))
    if panda_gl and len(panda_gl.dataset):
        pyt, pyp = graph_predict(_single, panda_gl); pv = bootstrap_qwk(pyt, pyp)
    results.append(("V5a Full (single fold)", sv[0], sv[1], pv[0], pv[1]))

# Variant 5b = full model, 5-fold ENSEMBLE (lower variance -> higher external QWK)
_ens = load_fold_ensemble()
if _ens:
    yt, yp, _ = ensemble_predict(_ens, val_gl); sv = bootstrap_qwk(yt, yp)
    pv = (None, (None, None))
    if panda_gl and len(panda_gl.dataset):
        pyt, pyp, _ = ensemble_predict(_ens, panda_gl); pv = bootstrap_qwk(pyt, pyp)
    results.append(("V5b Full Proposed (ens)", sv[0], sv[1], pv[0], pv[1]))

abl = pd.DataFrame([{
    "Variant": r[0],
    "SICAP_QWK": None if r[1] is None else round(r[1], 4),
    "SICAP_95CI": None if r[2] is None else (round(r[2][0], 3), round(r[2][1], 3)),
    "PANDA_QWK": None if r[3] is None else round(r[3], 4),
    "PANDA_95CI": None if r[4] is None else (round(r[4][0], 3), round(r[4][1], 3)),
} for r in results])
abl.to_csv(WORK / "ablation_results.csv", index=False)
print(abl.to_string(index=False))
print("Stage 14 complete.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

                Variant  SICAP_QWK     SICAP_95CI  PANDA_QWK      PANDA_95CI
     V1 Base Replicated     0.3279 (0.077, 0.547)     0.0808 (-0.004, 0.167)
        V2 +Multi-Scale     0.3543 (0.175, 0.501)     0.1914  (0.091, 0.285)
      V3 Graph+MeanPool     0.9332 (0.838, 0.979)     0.1063  (0.071, 0.144)
 V4 ILRA-only(no graph)     0.9281 (0.831, 0.976)     0.7736  (0.716, 0.825)
 V5a Full (single fold)     0.9543 (0.904, 0.983)     0.5644  (0.482, 0.644)
V5b Full Proposed (ens)     0.9219 (0.849, 0.965)     0.6571  (0.581, 0.727)
Stage 14 complete.


## Stage 14B — Pretrained encoder comparison

Swaps the frozen patch feature extractor for 5 ImageNet-pretrained backbones
(ResNet50, DenseNet121, EfficientNet-B0, ViT-B/16, ConvNeXt-Tiny) and runs the
**same** Graph-Transformer + ILRA head on each, versus the proposed DINOv2.
Backbones come from `torchvision` (no torch reinstall, so the P100 fix holds),
are loaded one-at-a-time to fit 16 GB, and run at 224x224.

In [17]:
import torchvision.models as tvm

RUN_CMP = True
CMP_BACKBONES = ["resnet50", "densenet121", "efficientnet_b0", "vit_b16", "convnext_tiny"]
CMP_EPOCHS = EPOCHS_GRAPH
cmp_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

def make_cmp_backbone(name):
    """Build an ImageNet-pretrained feature extractor -> (model, out_dim)."""
    if name == "resnet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2); d = m.fc.in_features; m.fc = nn.Identity()
    elif name == "densenet121":
        m = tvm.densenet121(weights=tvm.DenseNet121_Weights.IMAGENET1K_V1); d = m.classifier.in_features; m.classifier = nn.Identity()
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1); d = m.classifier[1].in_features; m.classifier = nn.Identity()
    elif name == "vit_b16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1); d = m.heads.head.in_features; m.heads = nn.Identity()
    elif name == "convnext_tiny":
        m = tvm.convnext_tiny(weights=tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); d = m.classifier[2].in_features; m.classifier[2] = nn.Identity()
    else:
        raise ValueError(name)
    m = m.to(DEVICE).eval()
    for p in m.parameters():
        p.requires_grad = False
    return m, d

@torch.no_grad()
def cmp_embed(model, paths, dim):
    """Embed a list of patch paths with a torchvision backbone -> (N, dim)."""
    out = []
    for i in range(0, len(paths), BATCH_SIZE_PATCH):
        xs = [cmp_tf(Image.open(p).convert("RGB")) for p in paths[i:i + BATCH_SIZE_PATCH]]
        x = torch.stack(xs).to(DEVICE)
        with torch.cuda.amp.autocast():
            f = model(x)
        out.append(f.float().cpu().numpy())
    return np.concatenate(out, 0) if out else np.zeros((0, dim), np.float32)

def cmp_extract_split(model, dim, name, csv_path, split):
    """Save per-slide 10x/20x embeddings + coords for one backbone/split."""
    base = WORK / f"embeddings/cmp/{name}/{split}"; base.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(csv_path); df["slide_id"] = df["filename"].apply(get_slide_id)
    for sid, g in tqdm(df.groupby("slide_id"), desc=f"emb {name}/{split}", leave=False):
        g = g.reset_index(drop=True)
        if (base / f"{sid}_10x.npy").exists():
            continue
        np.save(base / f"{sid}_10x.npy", cmp_embed(model, g["patch_path_10x"].tolist(), dim))
        np.save(base / f"{sid}_20x.npy", cmp_embed(model, g["patch_path_20x"].tolist(), dim))
        np.save(base / f"{sid}_coords.npy", np.array([get_coords(f) for f in g["filename"]], dtype=np.float32))
    return base

def graph_from_arrays(e10, e20, coords, isup, sid):
    """Build a multi-scale PyG graph from raw embedding/coord arrays."""
    N = min(len(e10), len(e20), len(coords))
    if N == 0:
        return None
    e10, e20, coords = e10[:N], e20[:N], coords[:N]
    cn = coords - coords.mean(0, keepdims=True); cn = cn / (np.abs(cn).max() + 1e-6)
    x10 = np.concatenate([e10, np.zeros((N, 1)), cn], 1)
    x20 = np.concatenate([e20, np.ones((N, 1)), cn], 1)
    x = torch.tensor(np.concatenate([x10, x20], 0), dtype=torch.float)
    ei10 = _knn_edges(coords, KNN_K); ei20 = _knn_edges(coords, KNN_K) + N
    cross = torch.stack([torch.arange(N), torch.arange(N) + N]); cross = torch.cat([cross, cross.flip(0)], 1)
    edge_index = torch.cat([ei10, ei20, cross], 1)
    return Data(x=x, edge_index=edge_index, y=torch.tensor([int(isup)], dtype=torch.long), slide_id=sid)

def cmp_build_graphs(name, split, slide_csv, cap=None):
    """Build graphs for one backbone/split from its saved embeddings."""
    base = WORK / f"embeddings/cmp/{name}/{split}"; out = WORK / f"graphs/cmp/{name}/{split}"
    out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(slide_csv)
    if cap:
        df = df.head(cap)
    for _, r in df.iterrows():
        sid = str(r["slide_id"])
        try:
            e10 = np.load(base / f"{sid}_10x.npy"); e20 = np.load(base / f"{sid}_20x.npy")
            coords = np.load(base / f"{sid}_coords.npy")
        except FileNotFoundError:
            continue
        g = graph_from_arrays(e10, e20, coords, int(r["isup_grade"]), sid)
        if g is not None:
            torch.save(g, out / f"{sid}.pt")
    return out

def cmp_train_eval(gtrain, gval, node_dim, tr_ids, va_ids, epochs):
    """Train GraphTransformerILRA on one backbone's graphs; return best val QWK."""
    tl = graph_loader(gtrain, True, True, tr_ids); vl = graph_loader(gval, False, False, va_ids)
    if len(tl.dataset) == 0 or len(vl.dataset) == 0:
        return None
    model = GraphTransformerILRA(node_feat_dim=node_dim, num_classes=graph_out_dim()).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(); best = -1
    for ep in range(epochs):
        model.train()
        for b in tl:
            b = b.to(DEVICE); opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = graph_loss(model(b), b.y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ys, ps = graph_predict(model, vl)
        q = cohen_kappa_score(ys, ps, weights="quadratic", labels=list(range(6))) if len(set(ys)) > 1 else 0.0
        best = max(best, q)
    return best

cmp_rows = []
tr_ids_cmp = pd.read_csv(WORK / "sicap_slide_labels_train.csv")["slide_id"].astype(str).tolist()
va_ids_cmp = pd.read_csv(WORK / "sicap_slide_labels_val.csv")["slide_id"].astype(str).tolist()

if RUN_CMP:
    # proposed DINOv2: reuse Stage 8 embeddings + Stage 10 graphs (apples-to-apples)
    q = cmp_train_eval(WORK / "graphs/sicap/train", WORK / "graphs/sicap/val", NODE_FEAT_DIM,
                       tr_ids_cmp, va_ids_cmp, CMP_EPOCHS)
    cmp_rows.append({"encoder": f"{ENCODER} (proposed)", "emb_dim": EMBED_DIM,
                     "SICAP_val_QWK": None if q is None else round(q, 4)})
    print(f"  {ENCODER} (proposed): SICAP val QWK = {q}")
    for name in CMP_BACKBONES:
        try:
            mdl, dim = make_cmp_backbone(name)
        except Exception as e:
            print("skip", name, e); continue
        cmp_extract_split(mdl, dim, name, TRAIN_CSV, "train")
        cmp_extract_split(mdl, dim, name, VAL_CSV, "val")
        del mdl; gc.collect(); torch.cuda.empty_cache()
        cmp_build_graphs(name, "train", WORK / "sicap_slide_labels_train.csv", MAX_GRAPH_SLIDES)
        cmp_build_graphs(name, "val", WORK / "sicap_slide_labels_val.csv", MAX_GRAPH_SLIDES)
        q = cmp_train_eval(WORK / f"graphs/cmp/{name}/train", WORK / f"graphs/cmp/{name}/val",
                           dim + 3, tr_ids_cmp, va_ids_cmp, CMP_EPOCHS)
        cmp_rows.append({"encoder": name, "emb_dim": dim,
                         "SICAP_val_QWK": None if q is None else round(q, 4)})
        print(f"  {name}: SICAP val QWK = {q}")
        gc.collect(); torch.cuda.empty_cache()

cmp_df = pd.DataFrame(cmp_rows).sort_values("SICAP_val_QWK", ascending=False, na_position="last")
cmp_df.to_csv(WORK / "backbone_comparison.csv", index=False)
print("\n=== Pretrained encoder comparison (SICAP val) ===")
print(cmp_df.to_string(index=False))
print("Stage 14B complete.")

  phikon (proposed): SICAP val QWK = 0.954269267668692
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]


emb resnet50/train:   0%|          | 0/366 [00:00<?, ?it/s]

emb resnet50/val:   0%|          | 0/33 [00:00<?, ?it/s]

  resnet50: SICAP val QWK = 0.8940852819807428
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 134MB/s]


... (11 outputs left out)

100%|██████████| 109M/109M [00:00<00:00, 164MB/s] 


emb convnext_tiny/train:   0%|          | 0/366 [00:00<?, ?it/s]

emb convnext_tiny/val:   0%|          | 0/33 [00:00<?, ?it/s]

  convnext_tiny: SICAP val QWK = 0.9329052969502407

=== Pretrained encoder comparison (SICAP val) ===
          encoder  emb_dim  SICAP_val_QWK
phikon (proposed)      768         0.9543
    convnext_tiny      768         0.9329
  efficientnet_b0     1280         0.9228
          vit_b16      768         0.9150
      densenet121     1024         0.8981
         resnet50     2048         0.8941
Stage 14B complete.


## Stage 15 — External validation on PANDA

In [18]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

_ens15 = load_fold_ensemble()
if _ens15 and panda_gl and len(panda_gl.dataset):
    # external prediction = AVERAGE of all 5 fold models (ensemble lowers variance)
    print(f"PANDA external: ensembling {len(_ens15)} fold models, objective={GRAPH_OBJECTIVE}")
    ys, ps, probs = ensemble_predict(_ens15, panda_gl)
    ys, ps = np.array(ys), np.array(ps)
    qwk, ci = bootstrap_qwk(ys, ps)
    acc = accuracy_score(ys, ps); mf1 = f1_score(ys, ps, average="macro")
    print(f"PANDA QWK={qwk:.4f} 95%CI={ci} acc={acc:.4f} macroF1={mf1:.4f}")

    # ---- threshold calibration (caveat #3): fit cut-points on SICAP val, lift accuracy ----
    qwk_cal = acc_cal = mf1_cal = None
    if GRAPH_OBJECTIVE == "ordinal" and val_gl and len(val_gl.dataset):
        vy, vs = ensemble_scores(_ens15, val_gl)
        py, pscore = ensemble_scores(_ens15, panda_gl)
        rounder = OptimizedRounder().fit(vs, vy)
        ps_cal = rounder.predict(pscore)
        qwk_cal, ci_cal = bootstrap_qwk(py, ps_cal)
        acc_cal = accuracy_score(py, ps_cal); mf1_cal = f1_score(py, ps_cal, average="macro")
        print(f"PANDA CALIBRATED QWK={qwk_cal:.4f} 95%CI={ci_cal} acc={acc_cal:.4f} macroF1={mf1_cal:.4f}")
        # keep whichever gives higher QWK as the headline
        if qwk_cal >= qwk:
            ps, qwk, ci, acc, mf1 = ps_cal, qwk_cal, ci_cal, acc_cal, mf1_cal

    # per-class AUROC (one-vs-rest)
    aurocs = {}
    for c in range(6):
        yb = (ys == c).astype(int)
        if yb.sum() and yb.sum() < len(yb):
            try:
                aurocs[c] = roc_auc_score(yb, probs[:, c])
            except Exception:
                aurocs[c] = float("nan")
    print("Per-class AUROC:", {k: round(v, 3) for k, v in aurocs.items()})

    cm = confusion_matrix(ys, ps, labels=list(range(6)))
    plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("PANDA confusion matrix")
    plt.savefig(WORK / "panda_confusion_matrix.png", dpi=100, bbox_inches="tight"); plt.close()

    plt.figure(figsize=(6, 4))
    plt.bar([str(k) for k in aurocs], list(aurocs.values()), color="steelblue")
    plt.ylim(0, 1); plt.xlabel("ISUP class"); plt.ylabel("AUROC"); plt.title("PANDA per-class AUROC")
    plt.savefig(WORK / "panda_auroc_per_class.png", dpi=100, bbox_inches="tight"); plt.close()
    PANDA_METRICS = {"qwk": qwk, "qwk_ci": ci, "acc": acc, "macro_f1": mf1,
                     "auroc_per_class": {str(k): v for k, v in aurocs.items()}}
else:
    PANDA_METRICS = {"note": "PANDA graphs unavailable in this run"}
    print(PANDA_METRICS["note"])
print("Stage 15 complete.")

PANDA external: ensembling 5 fold models, objective=ordinal
PANDA QWK=0.6571 95%CI=(0.5806131984338712, 0.7273635272286288) acc=0.4200 macroF1=0.3304
PANDA CALIBRATED QWK=0.6460 95%CI=(0.5692113278699542, 0.7153853867014263) acc=0.4267 macroF1=0.3449
Per-class AUROC: {0: np.float64(0.89), 1: np.float64(0.728), 2: np.float64(0.823), 3: np.float64(0.743), 4: np.float64(0.712), 5: np.float64(0.918)}
Stage 15 complete.


## Stage 16 — Attention heatmaps (ILRA cross-attention overlay)

In [19]:
from scipy.ndimage import gaussian_filter

def attention_heatmap(slide_id):
    """Overlay per-node ILRA attention importance on a PANDA thumbnail."""
    gp = WORK / f"graphs/panda/{slide_id}.pt"
    if not gp.exists():
        return False
    model = GraphTransformerILRA(num_classes=graph_out_dim()).to(DEVICE)
    model.load_state_dict(torch.load(FINAL_GRAPH_CKPT, map_location=DEVICE)); model.eval()
    data = torch.load(gp, weights_only=False).to(DEVICE)
    data.batch = torch.zeros(data.x.shape[0], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        _ = model(data)
    attn = model.ilra.last_attn[0].mean(0).cpu().numpy()   # (Nmax,)
    N = data.x.shape[0] // 2
    imp = attn[:N]                                          # 10x node importance
    coords = np.load(WORK / f"embeddings/panda/{slide_id}_coords.npy")[:N]
    H = int(coords[:, 0].max()) + 1; Wd = int(coords[:, 1].max()) + 1
    grid = np.zeros((H, Wd))
    for (rr, cc), v in zip(coords.astype(int), imp):
        grid[rr, cc] = v
    grid = gaussian_filter(grid, sigma=2)
    thumb = None
    try:
        import openslide
        s = openslide.OpenSlide(str(PANDA_IMG / f"{slide_id}.tiff"))
        thumb = np.array(s.read_region((0, 0), s.level_count - 1, s.level_dimensions[-1]).convert("RGB"))
        s.close()
    except Exception:
        pass
    fig, ax = plt.subplots(figsize=(5, 5))
    if thumb is not None:
        ax.imshow(thumb)
        gi = cv2.resize(grid, (thumb.shape[1], thumb.shape[0]))
        ax.imshow(gi, cmap="YlOrRd", alpha=0.5)
    else:
        ax.imshow(grid, cmap="YlOrRd")
    ax.set_title(slide_id); ax.axis("off")
    out = WORK / "heatmaps"; out.mkdir(exist_ok=True, parents=True)
    plt.savefig(out / f"{slide_id}_heatmap.png", dpi=90, bbox_inches="tight"); plt.close()
    return True

if FINAL_GRAPH_CKPT.exists() and (WORK / "panda_slide_labels.csv").exists():
    pl = pd.read_csv(WORK / "panda_slide_labels.csv")
    picks = []
    for g in range(1, 6):
        sub = pl[pl["isup_grade"] == g]
        if len(sub):
            picks.append(sub.iloc[0]["slide_id"])
    done = [s for s in picks if attention_heatmap(str(s))]
    print("Heatmaps saved for:", done)
else:
    print("Skipping heatmaps (no PANDA graphs / model).")
print("Stage 16 complete.")

Heatmaps saved for: ['003046e27c8ead3e3db155780dc5498e', '00bbc1482301d16de3ff63238cfd0b34', '0068d4c7529e34fd4c9da863ce01a161', '0018ae58b01bdadc8e347995b69f99aa', '00928370e2dfeb8a507667ef1d4efcbb']
Stage 16 complete.


## Stage 17 — Final results summary

In [20]:
summary = {"smoke_test": bool(SMOKE_TEST)}

# binary classifier metrics
try:
    bm = load_dinov2(2); bm.load_state_dict(torch.load(WORK / "binary_classifier_best.pt", map_location=DEVICE))
    _, vl = make_loaders("10x", "binary_label")
    _, m = evaluate(bm, vl, 2)
    summary["binary_classifier"] = {"val_auroc": m.get("auroc"), "val_acc": m["acc"], "val_qwk": m["qwk"]}
except Exception as e:
    summary["binary_classifier"] = {"error": str(e)}

# gp classifier metrics
try:
    gm = load_dinov2(3); gm.load_state_dict(torch.load(WORK / "gp_classifier_best.pt", map_location=DEVICE))
    _, gvl = make_loaders("10x", "gp_label", GP_TRAIN, GP_VAL)
    _, gmm = evaluate(gm, gvl, 3)
    summary["gp_classifier"] = {"val_qwk": gmm["qwk"], "val_acc": gmm["acc"], "val_auroc": gmm.get("auroc")}
except Exception as e:
    summary["gp_classifier"] = {"error": str(e)}

if "abl" in globals():
    summary["ablation"] = json.loads(abl.to_json(orient="records"))
if "cmp_df" in globals():
    summary["backbone_comparison"] = json.loads(cmp_df.to_json(orient="records"))
summary["panda_external"] = PANDA_METRICS if "PANDA_METRICS" in globals() else {}

def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    return o

with open(WORK / "final_results_summary.json", "w") as f:
    json.dump(_clean(summary), f, indent=2)
print(json.dumps(_clean(summary), indent=2))

print("\n=== Output artefacts in /kaggle/working ===")
for p in sorted(WORK.rglob("*")):
    if p.is_file() and p.suffix in (".pt", ".csv", ".png", ".json"):
        print(" ", p.relative_to(WORK))
print("\nStage 17 complete. Pipeline finished.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

{
  "smoke_test": false,
  "binary_classifier": {
    "val_auroc": 0.950167201419604,
    "val_acc": 0.8800925925925925,
    "val_qwk": 0.7603146448677411
  },
  "gp_classifier": {
    "val_qwk": 0.6746539815230471,
    "val_acc": 0.7974568574023615,
    "val_auroc": 0.8988822678438756
  },
  "ablation": [
    {
      "Variant": "V1 Base Replicated",
      "SICAP_QWK": 0.3279,
      "SICAP_95CI": [
        0.077,
        0.547
      ],
      "PANDA_QWK": 0.0808,
      "PANDA_95CI": [
        -0.004,
        0.167
      ]
    },
    {
      "Variant": "V2 +Multi-Scale",
      "SICAP_QWK": 0.3543,
      "SICAP_95CI": [
        0.175,
        0.501
      ],
      "PANDA_QWK": 0.1914,
      "PANDA_95CI": [
        0.091,
        0.285
      ]
    },
    {
      "Variant": "V3 Graph+MeanPool",
      "SICAP_QWK": 0.9332,
      "SICAP_95CI": [
        0.838,
        0.979
      ],
      "PANDA_QWK": 0.1063,
      "PANDA_95CI": [
        0.071,
        0.144
      ]
    },
    {
      "Variant

## Stage 18 — Publication figures & tables

Generates the journal-ready artefacts: confusion matrices (SICAP + PANDA),
per-class precision/recall/F1 + QWK tables, a paired-bootstrap significance test
(proposed vs base-paper rule), ILRA/attention visualisations, and a
parameter/runtime comparison table. All saved under `/kaggle/working/figures/`.

In [21]:
import time
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

PUB = WORK / "figures"; PUB.mkdir(parents=True, exist_ok=True)
ISUP_LABELS = [f"ISUP{i}" for i in range(NUM_ISUP)]
pubfig = {}

# ---- collect ENSEMBLE predictions (with slide ids) on SICAP val + PANDA ----
@torch.no_grad()
def ensemble_pred_dict(models, loader):
    """Return {slide_id: (true, pred)} from the fold ensemble."""
    out = {}
    if not models or loader is None or len(loader.dataset) == 0:
        return out
    for m in models:
        m.eval()
    for b in loader:
        b = b.to(DEVICE)
        with torch.cuda.amp.autocast():
            o = torch.stack([m(b).float() for m in models]).mean(0)
        preds = graph_pred(o).cpu().tolist(); trues = b.y.cpu().tolist()
        sids = b.slide_id if isinstance(b.slide_id, (list, tuple)) else [b.slide_id]
        for sid, t, p in zip(sids, trues, preds):
            out[str(sid)] = (int(t), int(p))
    return out

def aggregate_dict(split, multiscale=False):
    """Base-paper >5% rule predictions as {slide_id: (true, pred)}."""
    bm = load_dinov2(2); bm.load_state_dict(torch.load(WORK / "binary_classifier_best.pt", map_location=DEVICE)); bm.eval()
    gp = load_dinov2(3); gp.load_state_dict(torch.load(WORK / "gp_classifier_best.pt", map_location=DEVICE)); gp.eval()
    base = WORK / ("embeddings/panda" if split == "panda" else "embeddings/sicap/val")
    gtcsv = WORK / ("panda_slide_labels.csv" if split == "panda" else "sicap_slide_labels_val.csv")
    gt = pd.read_csv(gtcsv).set_index("slide_id")["isup_grade"].to_dict()
    out = {}
    with torch.no_grad():
        for sid, isup in gt.items():
            f10 = base / f"{sid}_10x.npy"
            if not f10.exists():
                continue
            t = torch.tensor(np.load(f10), dtype=torch.float, device=DEVICE)
            with torch.cuda.amp.autocast():
                mal = bm.head(t).argmax(1).cpu().numpy()
                gpr = torch.softmax(gp.head(t).float(), 1).cpu().numpy()
            gps = [["GP3", "GP4", "GP5"][int(gpr[i].argmax())] for i in range(len(mal)) if mal[i] == 1]
            if not gps:
                out[str(sid)] = (int(isup), 0); continue
            pct = pd.Series(gps).value_counts(); pct = pct / pct.sum() * 100
            gl = list(pct[pct > 5].sort_values(ascending=False).index)
            pr = gl[0] if gl else pd.Series(gps).value_counts().index[0]
            se = gl[1] if len(gl) > 1 else pr
            out[str(sid)] = (int(isup), ISUP_MAP.get((pr, se), ISUP_MAP.get((pr, pr), 1)))
    return out

_ens = load_fold_ensemble()
sic = ensemble_pred_dict(_ens, val_gl)
pan = ensemble_pred_dict(_ens, panda_gl) if panda_gl else {}

def _arrays(d):
    ks = sorted(d); return np.array([d[k][0] for k in ks]), np.array([d[k][1] for k in ks])

# ===== 1) CONFUSION MATRICES (SICAP val + PANDA) =====
panels = [("SICAP val", sic)] + ([("PANDA external", pan)] if pan else [])
fig, axs = plt.subplots(1, len(panels), figsize=(6 * len(panels), 5)); axs = np.atleast_1d(axs)
for ax, (title, d) in zip(axs, panels):
    if not d:
        continue
    y, p = _arrays(d)
    cm = confusion_matrix(y, p, labels=list(range(NUM_ISUP)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=ISUP_LABELS, yticklabels=ISUP_LABELS, cbar=False)
    qwk = cohen_kappa_score(y, p, weights="quadratic", labels=list(range(NUM_ISUP)))
    ax.set_title(f"{title} (QWK={qwk:.3f})"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig(PUB / "fig_confusion_matrices.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved fig_confusion_matrices.png")

# ===== 2) PER-CLASS metrics table (precision / recall / F1 / support + overall QWK/acc) =====
def per_class_rows(d, name):
    if not d:
        return []
    y, p = _arrays(d)
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=list(range(NUM_ISUP)), zero_division=0)
    rows = [{"dataset": name, "class": f"ISUP{c}", "precision": round(pr[c], 3),
             "recall": round(rc[c], 3), "f1": round(f1[c], 3), "support": int(sup[c])} for c in range(NUM_ISUP)]
    rows.append({"dataset": name, "class": "MACRO", "precision": round(pr.mean(), 3),
                 "recall": round(rc.mean(), 3), "f1": round(f1.mean(), 3), "support": int(sup.sum())})
    rows.append({"dataset": name, "class": "overall_acc",
                 "precision": round(accuracy_score(y, p), 3), "recall": "",
                 "f1": round(cohen_kappa_score(y, p, weights="quadratic", labels=list(range(NUM_ISUP))), 3),
                 "support": "QWK->f1col"})
    return rows

mtab = pd.DataFrame(per_class_rows(sic, "SICAP") + per_class_rows(pan, "PANDA"))
mtab.to_csv(PUB / "table_per_class_metrics.csv", index=False)
print("\n=== Per-class metrics (precision/recall/F1; last rows = macro + acc/QWK) ===")
print(mtab.to_string(index=False))

# ===== 3) STATISTICAL SIGNIFICANCE: proposed (ensemble) vs base rule (paired bootstrap) =====
def paired_bootstrap(y, pa, pb, n=N_BOOTSTRAP):
    """One-sided p that QWK(proposed a) > QWK(baseline b); returns (deltaQWK, p)."""
    y, pa, pb = np.array(y), np.array(pa), np.array(pb)
    rng = np.random.RandomState(SEED); diffs = []
    for _ in range(n):
        idx = rng.randint(0, len(y), len(y))
        if len(set(y[idx])) < 2:
            continue
        qa = cohen_kappa_score(y[idx], pa[idx], weights="quadratic", labels=list(range(NUM_ISUP)))
        qb = cohen_kappa_score(y[idx], pb[idx], weights="quadratic", labels=list(range(NUM_ISUP)))
        diffs.append(qa - qb)
    diffs = np.array(diffs)
    return (float(diffs.mean()), float((diffs <= 0).mean())) if len(diffs) else (0.0, 1.0)

sig_rows = []
for split, pdict in [("SICAP", sic), ("PANDA", pan)]:
    rdict = aggregate_dict("panda" if split == "PANDA" else "sicap_val")
    common = sorted(set(pdict) & set(rdict))
    if len(common) < 3:
        continue
    y = [pdict[s][0] for s in common]
    pa = [pdict[s][1] for s in common]      # proposed
    pb = [rdict[s][1] for s in common]      # base rule
    dq, pval = paired_bootstrap(y, pa, pb)
    sig_rows.append({"dataset": split, "n_slides": len(common),
                     "QWK_proposed": round(cohen_kappa_score(y, pa, weights="quadratic", labels=list(range(NUM_ISUP))), 3),
                     "QWK_rule": round(cohen_kappa_score(y, pb, weights="quadratic", labels=list(range(NUM_ISUP))), 3),
                     "delta_QWK": round(dq, 3), "p_value": round(pval, 4),
                     "significant_0.05": pval < 0.05})
sigdf = pd.DataFrame(sig_rows); sigdf.to_csv(PUB / "table_significance.csv", index=False)
print("\n=== Significance: proposed graph vs base-paper >5% rule (paired bootstrap) ===")
print(sigdf.to_string(index=False))

# ===== 4) ATTENTION VISUALISATION (ViT patch self-attention, encoder-specific) =====
def patch_attention_figure():
    """For HF ViT encoders (phikon): CLS->patch attention overlaid on example patches.
    Always draws the patches (so the figure has content even if attention is absent)
    and always closes the figure (no blank inline display)."""
    if ENCODER not in ("phikon", "phikon2"):
        print("  patch-attention: only for phikon encoders; see Stage 16 ILRA slide heatmaps.")
        return
    fig = None
    try:
        from transformers import AutoModel
        repo = "owkin/phikon" if ENCODER == "phikon" else "owkin/phikon-v2"
        try:
            hf = AutoModel.from_pretrained(repo, attn_implementation="eager").to(DEVICE).eval()
        except Exception:
            hf = AutoModel.from_pretrained(repo).to(DEVICE).eval()
        ex = pd.read_csv(VAL_CSV).groupby("label").head(1)   # one example per class
        n = max(1, len(ex)); fig, axs = plt.subplots(2, n, figsize=(3 * n, 6)); axs = np.atleast_2d(axs)
        tf = eval_tf(); drew_attn = 0
        for j, (_, r) in enumerate(ex.iterrows()):
            img = Image.open(r["patch_path_10x"]).convert("RGB")
            axs[0, j].imshow(img); axs[0, j].set_title(str(r["label"])); axs[0, j].axis("off")
            axs[1, j].imshow(img); axs[1, j].axis("off")
            try:
                x = tf(img).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    atts = getattr(hf(pixel_values=x, output_attentions=True), "attentions", None)
                if atts:
                    a = atts[-1][0].mean(0)[0, 1:]           # CLS->patches, head-averaged
                    s = int(a.shape[0] ** 0.5)
                    amap = a[: s * s].reshape(s, s).float().cpu().numpy()
                    amap = cv2.resize(amap, (img.width, img.height))
                    amap = (amap - amap.min()) / (np.ptp(amap) + 1e-8)
                    axs[1, j].imshow(amap, cmap="jet", alpha=0.45); drew_attn += 1
            except Exception:
                pass
        axs[0, 0].set_ylabel("patch"); axs[1, 0].set_ylabel("attention")
        plt.tight_layout(); plt.savefig(PUB / "fig_patch_attention.png", dpi=150, bbox_inches="tight")
        print(f"saved fig_patch_attention.png (attention overlaid on {drew_attn}/{n} patches)")
    except Exception as e:
        print("  patch-attention figure skipped:", e)
    finally:
        if fig is not None:
            plt.close(fig)

patch_attention_figure()

# ===== 5) PARAMETER & RUNTIME comparison table =====
def _params_m(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

rt_rows = []
try:
    enc_bb, enc_dim = build_backbone(ENCODER)
    rt_rows.append({"component": f"encoder:{ENCODER}", "params_M": round(_params_m(enc_bb), 2), "ms_per_item": ""})
    del enc_bb; gc.collect(); torch.cuda.empty_cache()
except Exception as e:
    print("encoder param count skipped:", e)

for label, pool, nl in [("GraphTransformer+ILRA (proposed)", "ilra", NUM_LAYERS),
                        ("Graph+MeanPool", "mean", NUM_LAYERS),
                        ("ILRA-only(no graph)", "ilra", 0)]:
    gm = GraphTransformerILRA(num_classes=graph_out_dim(), pool=pool, num_layers=nl).to(DEVICE).eval()
    ms = ""
    if val_gl and len(val_gl.dataset):
        b = next(iter(val_gl)).to(DEVICE)
        with torch.no_grad():
            for _ in range(2):
                gm(b)                      # warmup
            torch.cuda.synchronize(); t0 = time.time()
            for _ in range(10):
                gm(b)
            torch.cuda.synchronize()
        ms = round((time.time() - t0) / 10 / max(1, b.num_graphs) * 1000, 2)
    rt_rows.append({"component": label, "params_M": round(_params_m(gm), 3), "ms_per_item": ms})
    del gm; gc.collect(); torch.cuda.empty_cache()

rtdf = pd.DataFrame(rt_rows); rtdf.to_csv(PUB / "table_params_runtime.csv", index=False)
print("\n=== Parameter & runtime comparison ===")
print(rtdf.to_string(index=False))

print("\n=== Stage 18 artefacts in /kaggle/working/figures ===")
for p in sorted(PUB.glob("*")):
    print(" ", p.name)
print("Stage 18 complete: publication figures & tables generated.")

saved fig_confusion_matrices.png

=== Per-class metrics (precision/recall/F1; last rows = macro + acc/QWK) ===
dataset       class  precision recall    f1    support
  SICAP       ISUP0      1.000  0.778 0.875          9
  SICAP       ISUP1      0.333    0.2 0.250          5
  SICAP       ISUP2      0.500    1.0 0.667          3
  SICAP       ISUP3      0.500  0.667 0.571          3
  SICAP       ISUP4      0.000    0.0 0.000          1
  SICAP       ISUP5      0.857    0.5 0.632         12
  SICAP       MACRO      0.532  0.524 0.499         33
  SICAP overall_acc      0.576        0.922 QWK->f1col
  PANDA       ISUP0      0.463    1.0 0.633         75
  PANDA       ISUP1      0.391  0.225 0.286         80
  PANDA       ISUP2      0.478  0.289 0.361         38
  PANDA       ISUP3      0.343  0.308 0.324         39
  PANDA       ISUP4      0.226    0.2 0.212         35
  PANDA       ISUP5      1.000  0.091 0.167         33
  PANDA       MACRO      0.484  0.352 0.330        300
  PANDA o

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


=== Significance: proposed graph vs base-paper >5% rule (paired bootstrap) ===
dataset  n_slides  QWK_proposed  QWK_rule  delta_QWK  p_value  significant_0.05
  SICAP        33         0.922     0.328      0.597      0.0              True
  PANDA       300         0.657     0.081      0.576      0.0              True


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

saved fig_patch_attention.png (attention overlaid on 4/4 patches)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


=== Parameter & runtime comparison ===
                       component  params_M ms_per_item
                  encoder:phikon    86.390            
GraphTransformer+ILRA (proposed)    11.102        2.06
                  Graph+MeanPool     9.985        1.67
             ILRA-only(no graph)     1.645        0.48

=== Stage 18 artefacts in /kaggle/working/figures ===
  fig_confusion_matrices.png
  fig_patch_attention.png
  table_params_runtime.csv
  table_per_class_metrics.csv
  table_significance.csv
Stage 18 complete: publication figures & tables generated.


## Stage 18B — Grad-CAM & LIME explainability

Adds model-specific (**Grad-CAM**) and model-agnostic (**LIME**) saliency for the
GP classifier on example malignant patches, complementing the ViT attention
(Stage 18) and ILRA slide heatmaps (Stage 16). Saved to `/kaggle/working/figures/`.

In [22]:
EXPLAIN_EXAMPLES = 1                      # example patches per class
LIME_SAMPLES = 200 if SMOKE_TEST else 600  # LIME perturbations (more = sharper, slower)

def _load_gp_clf():
    clf = load_dinov2(3)
    clf.load_state_dict(torch.load(WORK / "gp_classifier_best.pt", map_location=DEVICE)); clf.eval()
    return clf

def _malignant_examples():
    ex = pd.read_csv(VAL_CSV)
    ex = ex[ex["binary_label"] == 1] if "binary_label" in ex else ex
    return ex.groupby("label").head(EXPLAIN_EXAMPLES)

def explain_gradcam():
    """Grad-CAM on the GP-classifier ViT (last block), overlaid on example patches."""
    fig = None
    try:
        from pytorch_grad_cam import GradCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
        clf = _load_gp_clf()
        bb = clf.backbone.m if hasattr(clf.backbone, "m") else clf.backbone  # HF ViTModel
        target = bb.encoder.layer[-1].layernorm_before                       # last ViT block
        def reshape_transform(t):
            s = int((t.size(1) - 1) ** 0.5)
            r = t[:, 1:1 + s * s, :].reshape(t.size(0), s, s, t.size(2))
            return r.permute(0, 3, 1, 2)
        cam = GradCAM(model=clf, target_layers=[target], reshape_transform=reshape_transform)
        ex = _malignant_examples(); tf = eval_tf()
        n = max(1, len(ex)); fig, axs = plt.subplots(1, n, figsize=(3 * n, 3)); axs = np.atleast_1d(axs)
        for j, (_, r) in enumerate(ex.iterrows()):
            img = Image.open(r["patch_path_10x"]).convert("RGB").resize((224, 224))
            g = cam(input_tensor=tf(img).unsqueeze(0).to(DEVICE))[0]
            vis = show_cam_on_image(np.asarray(img, np.float32) / 255.0, g, use_rgb=True)
            axs[j].imshow(vis); axs[j].set_title(str(r["label"])); axs[j].axis("off")
        plt.tight_layout(); plt.savefig(PUB / "fig_gradcam.png", dpi=150, bbox_inches="tight")
        print("saved fig_gradcam.png")
    except Exception as e:
        print("  Grad-CAM skipped:", e)
    finally:
        if fig is not None:
            plt.close(fig)

def explain_lime():
    """LIME (model-agnostic) superpixel attributions for the GP classifier."""
    fig = None
    try:
        from lime import lime_image
        from skimage.segmentation import mark_boundaries
        clf = _load_gp_clf(); tf = eval_tf()
        def predict(imgs):
            xs = torch.stack([tf(Image.fromarray(im.astype("uint8"))) for im in imgs]).to(DEVICE)
            with torch.no_grad(), torch.cuda.amp.autocast():
                return torch.softmax(clf(xs).float(), 1).cpu().numpy()
        ex = _malignant_examples()
        expl = lime_image.LimeImageExplainer()
        n = max(1, len(ex)); fig, axs = plt.subplots(1, n, figsize=(3 * n, 3)); axs = np.atleast_1d(axs)
        for j, (_, r) in enumerate(ex.iterrows()):
            img = np.asarray(Image.open(r["patch_path_10x"]).convert("RGB").resize((224, 224)))
            e = expl.explain_instance(img, predict, top_labels=1, hide_color=0, num_samples=LIME_SAMPLES)
            lab = e.top_labels[0]
            temp, mask = e.get_image_and_mask(lab, positive_only=True, num_features=6, hide_rest=False)
            axs[j].imshow(mark_boundaries(temp / 255.0, mask)); axs[j].set_title(str(r["label"])); axs[j].axis("off")
        plt.tight_layout(); plt.savefig(PUB / "fig_lime.png", dpi=150, bbox_inches="tight")
        print("saved fig_lime.png")
    except Exception as e:
        print("  LIME skipped:", e)
    finally:
        if fig is not None:
            plt.close(fig)

explain_gradcam()
explain_lime()
print("Stage 18B complete: Grad-CAM + LIME explainability.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

saved fig_gradcam.png


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/600 [00:00<?, ?it/s]

saved fig_lime.png
Stage 18B complete: Grad-CAM + LIME explainability.
